# CORnet E/I Imbalance: Face Recognition under Simulated ASD Pathology

This notebook runs the full experimental pipeline:

1. **Setup** — install dependencies, mount Drive, import modules
2. **Post-hoc analysis** — inspect previously saved CORnet results
3. **Dataset preparation** — build balanced VGGFace2 / LFW subsets
4. **Experimental conditions** — define E/I alpha values and inspect model architecture
5. **Reproducibility & utilities** — seed everything, define saving helpers
6. **Main training runs** (10 runs x 3 conditions x 100 epochs)
7. **Visualization & export** — training curves, RSA heatmaps, pickle export
8. **Pilot / parameter-search runs** — varying dataset size, batch size, decoder width, dropout


## 1. Environment Setup


In [5]:
# Check if running on Google Colab
!pip install -q git+https://github.com/dicarlolab/CORnet

from google.colab import drive
import sys
import os

drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/MyDrive/ASD_FaceReg_Modeling_CNN'
%cd {PROJECT_PATH}

!nvidia-smi

if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)

print(f"Working directory set to: {os.getcwd()}")

  Preparing metadata (setup.py) ... done
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/ASD_FaceReg_Modeling_CNN
/bin/bash: line 1: nvidia-smi: command not found
Working directory set to: /content/drive/MyDrive/ASD_FaceReg_Modeling_CNN


### 1.1 Imports


In [6]:
# ==============================
# Standard Library
# ==============================
import sys
import random
import json
import re
import shutil
import subprocess
import time
from collections import defaultdict

# ==============================
# Third-Party Libraries
# ==============================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# ==============================
# PyTorch / TorchVision
# ==============================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torchvision.io import read_image, ImageReadMode
from torch.utils.data import DataLoader, random_split, TensorDataset, Subset

# ==============================
# Local Project Imports
# ==============================
# Ensure project root is importable
if "." not in sys.path:
    sys.path.insert(0, ".")

# Data loading and setup
from src.data import (
    # Loader: download and prepare datasets
    prepare_balanced_lfw,
    validate_dataset,
    check_dataset_exists,
    # Preprocessing: load data for training
    get_dataloader,
    get_split_dataloaders,
    get_processed_dataloader,
    get_processed_split_dataloaders,
)

# Model training
from src.models.cornet import build_cornet_for_training, train_cornet

# Visualization
from src.analysis import visualize_training_comparisons

# RSA analysis
from src.analysis import (
    compute_rsa_matrix,
    aggregate_and_visualize_rsa,
    visualize_rsa_results,
    get_person_name_labels,
    visualize_local_results,
)

# Utils
from src.utils import get_device, empty_cache

# ==============================
# Sanity Check
# ==============================
print("All modules imported!")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")


All modules imported!
PyTorch: 2.9.0+cpu
CUDA: False


## 2. Post-hoc Analysis of Saved Results

Load and inspect results from a previous CORnet training experiment.
These cells read already-saved JSON / `.npy` outputs from Google Drive — no model is trained here.


### 2.1 Load Experiment Folder


In [ ]:

EXP_FOLDER_NAME = "100_20_100_100_2026-02-07_03-11-07"

run_dir_path = os.path.join(
    "/content/drive/MyDrive/ASD_FaceReg_Modeling_CNN/results/EIB/cornet",
    EXP_FOLDER_NAME,
    "run_0"
)


summary_stats = visualize_local_results(
    run_dir=run_dir_path,
    conditions=["Balanced", "Excitated", "Inhibitated"],
    show_training_curves=True,
    show_rsa_matrices=True,
    full_label=False,
)


print("\nReturned Summary Data:")
print(summary_stats)

Output hidden; open in https://colab.research.google.com to view.

### 2.2 Inspect Directory


In [9]:
!pwd

/content/drive/MyDrive/ASD_FaceReg_Modeling_CNN


In [12]:
import os

EXP_DIR = "/content/drive/MyDrive/ASD_FaceReg_Modeling_CNN/results/EIB/cornet/100_20_100_100_2026-02-07_03-11-07"

# Check if base dir exists
print("EXP_DIR exists:", os.path.isdir(EXP_DIR))

# List top level
if os.path.isdir(EXP_DIR):
    print("\nTop level:", sorted(os.listdir(EXP_DIR))[:10])

# List run_0 contents
run0 = os.path.join(EXP_DIR, "run_0")
if os.path.isdir(run0):
    print("\nrun_0 contents:", sorted(os.listdir(run0)))

    # List first subfolder contents
    for d in sorted(os.listdir(run0)):
        full = os.path.join(run0, d)
        if os.path.isdir(full):
            print(f"\n{d}/:", sorted(os.listdir(full))[:8])
            break

EXP_DIR exists: True

Top level: ['output', 'run_0', 'run_1', 'run_10', 'run_11', 'run_12', 'run_13', 'run_14', 'run_15', 'run_16']

run_0 contents: ['Balanced', 'Balanced_test_matrices.npz', 'Balanced_train_matrices.npz', 'Excitated', 'Excitated_test_matrices.npz', 'Excitated_train_matrices.npz', 'Inhibitated', 'Inhibitated_test_matrices.npz', 'Inhibitated_train_matrices.npz', 'all_raw_histories.json', 'test_labels_by_condition.json', 'train_labels_by_condition.json']

Balanced/: ['history_run0.json', 'model_run0.pt', 'rsa_test_labels_run0.json', 'rsa_test_run0.npy', 'rsa_train_labels_run0.json', 'rsa_train_run0.npy']


### 2.3 Extract Within- vs Between-Person RSA Correlations

Parse the saved RSA `.npy` matrices, compute within-person and between-person mean correlations, and summarise per condition.


In [17]:
"""
Extract within-person and between-person mean correlations
from RSA .npy files (both train and test splits).

STEP 1: Copy all .npy files from Google Drive to Colab local SSD
STEP 2: Process from local disk (10-50x faster IO)

Output: {OUTPUT_DIR}/nrs_within_between.csv
"""

import os
import shutil
import csv
import numpy as np
from tqdm import tqdm

# ============================================================
# CONFIG
# ============================================================
GDRIVE_EXP_DIR = "/content/drive/MyDrive/ASD_FaceReg_Modeling_CNN/results/EIB/cornet/100_20_100_100_2026-02-07_03-11-07"
LOCAL_CACHE     = "/content/rsa_cache"   # Colab SSD — fast!

OUTPUT_DIR = os.path.join(GDRIVE_EXP_DIR, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_RUNS   = 20
N_PEOPLE = 100

CONDITIONS = [
    {"name": "Inhibitated", "folder": "Inhibitated", "slope": 0.5},
    {"name": "Balanced",    "folder": "Balanced",     "slope": 1.0},
    {"name": "Excitated",   "folder": "Excitated",    "slope": 2.0},
]

SPLITS = ["train", "test"]

# ============================================================
# STEP 1: Copy .npy files to local SSD
# ============================================================
print("=" * 60)
print("STEP 1: Copying .npy files to local SSD")
print("=" * 60)

files_to_copy = []
for split in SPLITS:
    for cond in CONDITIONS:
        for run_idx in range(N_RUNS):
            rel_path = os.path.join(
                f"run_{run_idx}", cond["folder"],
                f"rsa_{split}_run{run_idx}.npy"
            )
            src = os.path.join(GDRIVE_EXP_DIR, rel_path)
            dst = os.path.join(LOCAL_CACHE, rel_path)
            files_to_copy.append((src, dst))

# Check first file exists
if not os.path.exists(files_to_copy[0][0]):
    # Try to find actual structure
    print(f"\nERROR: {files_to_copy[0][0]} not found!")
    print("\nDiagnostic — listing actual structure:")
    run0 = os.path.join(GDRIVE_EXP_DIR, "run_0")
    if os.path.isdir(run0):
        for d in sorted(os.listdir(run0)):
            full = os.path.join(run0, d)
            if os.path.isdir(full):
                print(f"  {d}/: {sorted(os.listdir(full))[:5]}")
    else:
        print(f"  run_0 dir not found. Top level: {sorted(os.listdir(GDRIVE_EXP_DIR))[:10]}")
    raise SystemExit("Fix the paths above and re-run.")

# Copy with progress bar
already_cached = 0
to_copy = 0

for src, dst in files_to_copy:
    if os.path.exists(dst):
        already_cached += 1
    else:
        to_copy += 1

print(f"\nTotal files: {len(files_to_copy)}")
print(f"Already cached: {already_cached}")
print(f"To copy: {to_copy}")

if to_copy > 0:
    print("\nCopying from Google Drive → local SSD...")
    for src, dst in tqdm(files_to_copy, desc="Copying"):
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)
    print("Copy complete!")
else:
    print("All files already cached — skipping copy.")

# ============================================================
# STEP 2: Process from local SSD
# ============================================================
print("\n" + "=" * 60)
print("STEP 2: Extracting within/between from local .npy files")
print("=" * 60)

def build_masks(n_total, n_people):
    imgs_pp = n_total // n_people
    rows, cols = np.tril_indices(n_total, k=-1)
    p_row = rows // imgs_pp
    p_col = cols // imgs_pp
    within  = (p_row == p_col)
    between = ~within
    print(f"  {n_total}×{n_total}, {imgs_pp} imgs/person, "
          f"{within.sum():,} within, {between.sum():,} between")
    return rows, cols, within, between

results = []

for split in SPLITS:
    print(f"\n--- {split.upper()} RSA ---")

    # Detect size
    first_path = os.path.join(LOCAL_CACHE, "run_0", CONDITIONS[0]["folder"],
                               f"rsa_{split}_run0.npy")
    n_total = np.load(first_path).shape[0]
    rows, cols, w_mask, b_mask = build_masks(n_total, N_PEOPLE)

    for cond in CONDITIONS:
        desc = f"{cond['name']} ({split})"
        for run_idx in tqdm(range(N_RUNS), desc=f"  {desc}"):
            path = os.path.join(LOCAL_CACHE, f"run_{run_idx}", cond["folder"],
                                f"rsa_{split}_run{run_idx}.npy")
            mat = np.load(path)
            vals = mat[rows, cols]

            results.append({
                "Sub": f"Sub{run_idx + 1}",
                "slope": cond["slope"],
                "condition": cond["name"],
                "Type": "Within",
                "corr": float(np.nanmean(vals[w_mask])),
                "split": split,
            })
            results.append({
                "Sub": f"Sub{run_idx + 1}",
                "slope": cond["slope"],
                "condition": cond["name"],
                "Type": "Between",
                "corr": float(np.nanmean(vals[b_mask])),
                "split": split,
            })
            del mat, vals

# ============================================================
# Save CSV (to Google Drive so R can read it)
# ============================================================
out_path = os.path.join(OUTPUT_DIR, "nrs_within_between.csv")
with open(out_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["Sub","slope","condition","Type","corr","split"])
    writer.writeheader()
    writer.writerows(results)

print(f"\n{'=' * 60}")
print(f"Done! {len(results)} rows saved to:")
print(f"  {out_path}")
print(f"\nNow run Repres_data_EIB.R in RStudio.")
print(f"{'=' * 60}")

STEP 1: Copying .npy files to local SSD

Total files: 120
Already cached: 0
To copy: 120

Copying from Google Drive → local SSD...


Copying: 100%|██████████| 120/120 [17:08<00:00,  8.57s/it]


Copy complete!

STEP 2: Extracting within/between from local .npy files

--- TRAIN RSA ---
  8000×8000, 80 imgs/person, 316,000 within, 31,680,000 between


  Excitated (train): 100%|██████████| 20/20 [00:11<00:00,  1.71it/s]



--- TEST RSA ---
  1000×1000, 10 imgs/person, 4,500 within, 495,000 between


  Excitated (test): 100%|██████████| 20/20 [00:00<00:00, 245.35it/s]



Done! 240 rows saved to:
  /content/drive/MyDrive/ASD_FaceReg_Modeling_CNN/results/EIB/cornet/100_20_100_100_2026-02-07_03-11-07/output/nrs_within_between.csv

Now run Repres_data_EIB.R in RStudio.


## 3. Dataset Preparation


### 3.1 VGGFace2 Balanced Subset

Create a balanced face-image dataset from VGGFace2: select identities, copy images, and build the folder structure expected by the PyTorch `DataLoader`.


In [ ]:
# ==========================================
# Imports
# ==========================================
import os
import shutil
import subprocess
import copy
from collections import defaultdict

import torch
from torch.utils.data import Subset
from torchvision import datasets, transforms


# ==========================================
# Configuration
# ==========================================
DRIVE_ROOT = "/content/drive/MyDrive/ASD_FaceReg_Modeling_CNN/data/vggface2/data"
DRIVE_TRAIN_TAR = f"{DRIVE_ROOT}/vggface2_train.tar.gz"
DRIVE_TEST_TAR = f"{DRIVE_ROOT}/vggface2_test.tar.gz"

LOCAL_ROOT = "/content/vggface2"
LOCAL_TRAIN_DIR = f"{LOCAL_ROOT}/train"
LOCAL_TEST_DIR = f"{LOCAL_ROOT}/test"

# Balanced dataset save (optional)
SAVE_BALANCED_TO_DRIVE = False
BALANCED_DRIVE_BASE = "/content/drive/MyDrive/ASD_FaceReg_Modeling_CNN/data"

# Reproducibility
GLOBAL_SEED = 42

# Hyperparameters
N_PEOPLE = 100
IMGS_PER_PERSON = 100
BATCH_SIZE = 2048
NUM_WORKERS = 2

# Split ratios (per class)
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

# ==========================================
# 0. Data Preparation Switch (BALANCED -> SSD) OR (TAR -> SSD)
# ==========================================
# Option A (Logic 1): Provide balanced dataset directory on Drive (local dataset)
#   - If this is set to a path: we REQUIRE it exists and is valid, else raise error.
#   - If this is None: we will NOT try to use balanced dir.
BALANCED_DRIVE_DIR = f"/content/drive/MyDrive/ASD_FaceReg_Modeling_CNN/data/vggface2_balanced_{N_PEOPLE}_{IMGS_PER_PERSON}"
# If you want to disable Logic 1 and force tar extraction, set:
# BALANCED_DRIVE_DIR = None

# SSD target for balanced copy (only used if BALANCED_DRIVE_DIR is not None)
SSD_BALANCED_DIR = f"/content/vggface2_balanced_{N_PEOPLE}_{IMGS_PER_PERSON}"


def _count_images_under(root):
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    total = 0
    for _, _, fns in os.walk(root):
        total += sum(fn.lower().endswith(exts) for fn in fns)
    return total


def _assert_is_valid_imagefolder(root, what="dataset"):
    """
    Strict validation:
      - root exists and is a directory
      - contains at least 1 image (jpg/png/webp/bmp)
    """
    if root is None:
        raise ValueError(f"{what} path is None.")
    if not os.path.isdir(root):
        raise FileNotFoundError(f"{what} directory not found: {root}")
    n = _count_images_under(root)
    if n <= 0:
        raise RuntimeError(f"{what} directory has no image files: {root}")
    return n

def _copy_dir_drive_to_ssd_strict(src_dir, dst_dir):
    """
    Drive folder -> SSD folder via system 'cp -r'.
    Much faster and more stable than python shutil or tar pipes on Colab.
    """
    src_total = _assert_is_valid_imagefolder(src_dir, what="Balanced Drive dataset")

    # Idempotency: if dst exists and counts match -> skip
    if os.path.isdir(dst_dir):
        dst_total = _count_images_under(dst_dir)
        if dst_total == src_total:
            print(f"[SSD] Balanced dataset already on SSD: {dst_dir} (skip copying)")
            return
        else:
            # If counts don't match, remove partial copy to ensure clean state
            print(f"[SSD] Mismatch found (src={src_total}, dst={dst_total}). Re-copying...")
            shutil.rmtree(dst_dir)

    print(f"[COPY] Copying Drive -> SSD via system 'cp' (fast)...\n  src: {src_dir}\n  dst: {dst_dir}")

    # Ensure source path doesn't end in slash for consistent 'cp' behavior
    src_dir = src_dir.rstrip("/")

    # We use 'cp -r' which is optimized at the OS level.
    # If dst_dir does not exist, 'cp -r src dst' creates dst and copies contents of src into it.
    try:
        subprocess.run(["cp", "-r", src_dir, dst_dir], check=True)
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"[COPY] System copy failed: {e}")

    # Strict verify
    dst_total = _count_images_under(dst_dir)
    if dst_total != src_total:
        raise RuntimeError(
            f"[SSD] Copy incomplete: src={src_total}, dst={dst_total}\n"
            f"  src_dir={src_dir}\n  dst_dir={dst_dir}"
        )

    print("[SSD] Done.")

def _extract_train_tar_to_ssd_strict(drive_tar, local_root, expected_train_dir):
    """
    Extract tar.gz from Drive to SSD with strict correctness:
      - tar must exist
      - if expected_train_dir already exists & has images -> skip
      - else extract
      - verify expected_train_dir exists and contains images
    """
    if not os.path.isfile(drive_tar):
        raise FileNotFoundError(f"Train tar.gz not found: {drive_tar}")

    # idempotency: if already extracted and has images -> skip
    if os.path.isdir(expected_train_dir) and _count_images_under(expected_train_dir) > 0:
        print(f"[TAR] Train folder already ready: {expected_train_dir} (skip extracting)")
        return

    print(f"[TAR] Extracting train tar -> SSD\n  src: {drive_tar}\n  dst root: {local_root}")
    os.makedirs(local_root, exist_ok=True)

    # extract
    subprocess.run(["tar", "-xzf", drive_tar, "-C", local_root], check=True)

    # strict verify
    if not os.path.isdir(expected_train_dir):
        raise RuntimeError(
            f"[TAR] Expected train dir not found after extract: {expected_train_dir}\n"
            f"Check tar structure under: {local_root}"
        )
    n = _count_images_under(expected_train_dir)
    if n <= 0:
        raise RuntimeError(f"[TAR] Extracted train dir has no images: {expected_train_dir}")
    print("[TAR] Done.")


# ---- Decide logic ----
# 1) If BALANCED_DRIVE_DIR is provided (not None) => MUST copy to SSD; wrong => error
# 2) Else => MUST extract from DRIVE_TRAIN_TAR; wrong => error
if BALANCED_DRIVE_DIR is not None:
    # Logic 1
    _copy_dir_drive_to_ssd_strict(BALANCED_DRIVE_DIR, SSD_BALANCED_DIR)
    # Keep naming: point LOCAL_TRAIN_DIR to SSD balanced
    LOCAL_TRAIN_DIR = SSD_BALANCED_DIR
    print(f"[PATH] Using balanced dataset on SSD -> LOCAL_TRAIN_DIR = {LOCAL_TRAIN_DIR}")
else:
    # Logic 2
    _extract_train_tar_to_ssd_strict(DRIVE_TRAIN_TAR, LOCAL_ROOT, LOCAL_TRAIN_DIR)
    print(f"[PATH] Using extracted raw train on SSD -> LOCAL_TRAIN_DIR = {LOCAL_TRAIN_DIR}")


# ==========================================
# 1. Configuration & Device
# ==========================================
def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

DEVICE = get_device()
print(f"Device: {DEVICE}")


# ==========================================
# 2. Data Transforms
# ==========================================
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

data_transforms = {
    "train": transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(NORM_MEAN, NORM_STD),
    ]),
    "eval": transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(NORM_MEAN, NORM_STD),
    ]),
}


# ==========================================
# 3. Data Loading & STRICT Balanced Filtering
# ==========================================
def _maybe_save_balanced_to_drive(save_dir, class_names, samples):
    """
    Save the balanced dataset to Google Drive:
      save_dir/
        <class_name>/
          <image files...>
    """
    expected_total = len(samples)

    # Fast idempotency check
    if os.path.exists(save_dir):
        total_files = 0
        ok_structure = True
        for cname in class_names:
            cdir = os.path.join(save_dir, cname)
            if not os.path.isdir(cdir):
                ok_structure = False
                break
            total_files += sum(
                1 for fn in os.listdir(cdir)
                if fn.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
            )
        if ok_structure and total_files == expected_total:
            print(f"[SAVE] Balanced dataset already exists: {save_dir} (skip copying)")
            return

    print(f"[SAVE] Copying balanced dataset to: {save_dir}")
    os.makedirs(save_dir, exist_ok=True)

    # Copy images
    for path, new_label in samples:
        class_name = class_names[new_label]
        class_dir = os.path.join(save_dir, class_name)
        os.makedirs(class_dir, exist_ok=True)

        dst = os.path.join(class_dir, os.path.basename(path))
        if not os.path.exists(dst):
            shutil.copy2(path, dst)

    print("[SAVE] Done.")


def get_filtered_dataset(root_dir, transform, n_classes, max_imgs_per_class):
    """
    Loads dataset and STRICTLY filters:
      - Select ONLY classes with >= max_imgs_per_class images ("qualified")
      - Select EXACTLY n_classes classes (alphabetical order by folder name)
      - Take EXACTLY max_imgs_per_class images per class (deterministic: sorted paths)
      - Remap labels to 0..n_classes-1 and update classes/class_to_idx accordingly

    Optional:
      - Save the balanced dataset to Drive when SAVE_BALANCED_TO_DRIVE=True
    """
    print(f"Loading raw data from: {root_dir}")
    raw = datasets.ImageFolder(root=root_dir, transform=transform)

    # Group paths by original label
    label_to_paths = defaultdict(list)
    for path, lbl in raw.samples:
        label_to_paths[lbl].append(path)

    # Find qualified classes (must have enough images)
    qualified_old_labels = []
    for old_lbl in range(len(raw.classes)):  # keep original alphabetical class order
        if len(label_to_paths.get(old_lbl, [])) >= max_imgs_per_class:
            qualified_old_labels.append(old_lbl)

    if len(qualified_old_labels) < n_classes:
        raise ValueError(
            f"Not enough qualified classes. Need {n_classes}, but found {len(qualified_old_labels)} "
            f"with >= {max_imgs_per_class} images."
        )

    # Select first N qualified classes (still alphabetical by folder name)
    selected_old_labels = qualified_old_labels[:n_classes]
    selected_class_names = [raw.classes[old_lbl] for old_lbl in selected_old_labels]

    # Remap old labels -> new labels (0..n_classes-1)
    old_to_new = {old_lbl: new_lbl for new_lbl, old_lbl in enumerate(selected_old_labels)}

    # Build STRICT balanced samples (exactly K per class)
    new_samples = []
    for old_lbl in selected_old_labels:
        paths = sorted(label_to_paths[old_lbl])[:max_imgs_per_class]
        new_lbl = old_to_new[old_lbl]
        new_samples.extend([(p, new_lbl) for p in paths])

    # Overwrite dataset internals to reflect the balanced subset
    raw.samples = new_samples
    raw.imgs = new_samples
    raw.targets = [lbl for _, lbl in new_samples]
    raw.classes = selected_class_names
    raw.class_to_idx = {name: i for i, name in enumerate(selected_class_names)}

    # Sanity check
    total_expected = n_classes * max_imgs_per_class
    if len(raw) != total_expected:
        raise RuntimeError(
            f"Balanced dataset size mismatch. Expected {total_expected}, got {len(raw)}."
        )

    # Optional save
    if SAVE_BALANCED_TO_DRIVE:
        save_dir = os.path.join(
            BALANCED_DRIVE_BASE,
            f"vggface2_balanced_{n_classes}_{max_imgs_per_class}",
        )
        _maybe_save_balanced_to_drive(save_dir, raw.classes, raw.samples)

    print(f"Filtered Total Size: {len(raw)} images "
          f"({n_classes} classes × {max_imgs_per_class} imgs/class)")
    return raw


# Build a balanced dataset with deterministic transform (good for val/test + RSA)
full_dataset = get_filtered_dataset(
    LOCAL_TRAIN_DIR,
    data_transforms["eval"],
    N_PEOPLE,
    IMGS_PER_PERSON
)


# ==========================================
# 4. STRICT Stratified Split (per-class)
# ==========================================
def _build_stratified_indices(targets, imgs_per_class, train_ratio, val_ratio, seed):
    """
    Split indices PER CLASS so each class has identical counts in train/val/test.
    """
    if abs(train_ratio + val_ratio - 1.0 + TEST_RATIO) > 1e-6:
        # We still compute test as remainder; this is just a safeguard for common mistakes.
        pass

    train_k = int(imgs_per_class * train_ratio)
    val_k = int(imgs_per_class * val_ratio)
    test_k = imgs_per_class - train_k - val_k

    if train_k <= 0 or val_k <= 0 or test_k <= 0:
        raise ValueError(
            f"Invalid split counts per class: train={train_k}, val={val_k}, test={test_k}. "
            f"Please adjust ratios or IMGS_PER_PERSON."
        )

    per_class_indices = defaultdict(list)
    for idx, y in enumerate(targets):
        per_class_indices[int(y)].append(idx)

    # Sanity: each class must have exactly imgs_per_class
    for c, idxs in per_class_indices.items():
        if len(idxs) != imgs_per_class:
            raise RuntimeError(
                f"Class {c} has {len(idxs)} samples, expected {imgs_per_class}. "
                f"Your balanced filtering did not enforce strict per-class counts."
            )

    g = torch.Generator().manual_seed(seed)

    train_indices, val_indices, test_indices = [], [], []
    for c in sorted(per_class_indices.keys()):
        idxs = per_class_indices[c]
        perm = torch.randperm(len(idxs), generator=g).tolist()
        idxs = [idxs[i] for i in perm]

        train_indices.extend(idxs[:train_k])
        val_indices.extend(idxs[train_k:train_k + val_k])
        test_indices.extend(idxs[train_k + val_k:train_k + val_k + test_k])

    return train_indices, val_indices, test_indices, train_k, val_k, test_k


train_indices, val_indices, test_indices, train_k, val_k, test_k = _build_stratified_indices(
    full_dataset.targets,
    IMGS_PER_PERSON,
    TRAIN_RATIO,
    VAL_RATIO,
    GLOBAL_SEED
)

print("   Splitting strategy (STRICT per-class):")
print(f"   Per class -> Train: {train_k}, Val: {val_k}, Test: {test_k}")
print(f"   Total    -> Train: {len(train_indices)}, Val: {len(val_indices)}, Test: {len(test_indices)}")


# Create a train-view dataset with augmentation, sharing the same samples/labels
train_dataset = copy.copy(full_dataset)
train_dataset.transform = data_transforms["train"]

# Final outputs (keep names unchanged)
train_set = Subset(train_dataset, train_indices)
val_set = Subset(full_dataset, val_indices)
test_set = Subset(full_dataset, test_indices)


# ==========================================
# (Optional) Quick verification print
# ==========================================
def _count_per_class(subset, n_classes):
    counts = [0] * n_classes
    for idx in subset.indices:
        y = int(subset.dataset.targets[idx])
        counts[y] += 1
    return min(counts), max(counts), sorted(set(counts))

mn, mx, uniq = _count_per_class(train_set, N_PEOPLE)
print(f"[CHECK] Train per-class count -> min={mn}, max={mx}, unique={uniq}")

mn, mx, uniq = _count_per_class(val_set, N_PEOPLE)
print(f"[CHECK] Val   per-class count -> min={mn}, max={mx}, unique={uniq}")

mn, mx, uniq = _count_per_class(test_set, N_PEOPLE)
print(f"[CHECK] Test  per-class count -> min={mn}, max={mx}, unique={uniq}")


[COPY] Copying Drive -> SSD via system 'cp' (fast)...
  src: /content/drive/MyDrive/ASD_FaceReg_Modeling_CNN/data/vggface2_balanced_100_100
  dst: /content/vggface2_balanced_100_100
[SSD] Done.
[PATH] Using balanced dataset on SSD -> LOCAL_TRAIN_DIR = /content/vggface2_balanced_100_100
Device: cuda
Loading raw data from: /content/vggface2_balanced_100_100
Filtered Total Size: 10000 images (100 classes × 100 imgs/class)
   Splitting strategy (STRICT per-class):
   Per class -> Train: 80, Val: 10, Test: 10
   Total    -> Train: 8000, Val: 1000, Test: 1000
[CHECK] Train per-class count -> min=80, max=80, unique=[80]
[CHECK] Val   per-class count -> min=10, max=10, unique=[10]
[CHECK] Test  per-class count -> min=10, max=10, unique=[10]


### 3.2 LFW Subset

Prepare a smaller evaluation subset from LFW (Labeled Faces in the Wild).


In [ ]:
# ==========================================
# 1. Configuration(lfw)
# ==========================================
DATA_DIR = "/content/drive/MyDrive/ASD_FaceReg_Modeling_CNN/data/lfw/lfw-deepfunneled/lfw-deepfunneled"
BALANCED_DATA_DIR = "/content/drive/MyDrive/ASD_FaceReg_Modeling_CNN/data/lfw/balanced_50_30"

N_PEOPLE = 50
IMGS_PER_PERSON = 30
BATCH_SIZE = 256
DEVICE = get_device()


# ==========================================
# 2. Data Preparation
# ==========================================
prepare_balanced_lfw(
    save_dir=BALANCED_DATA_DIR,
    n_people=N_PEOPLE,
    imgs_per_person=IMGS_PER_PERSON,
    local_lfw_dir=DATA_DIR
)
validate_dataset(BALANCED_DATA_DIR)

loaders = get_split_dataloaders(
    data_dir=BALANCED_DATA_DIR,
    batch_size=BATCH_SIZE,
    split_ratio=(0.8, 0.1, 0.1),
    num_workers=0,
    seed=42
)

train_loader = loaders['train']
val_loader = loaders['val']
test_loader = loaders['test']

print(f"\nData loading complete!")
print(f"  Train: {len(train_loader.dataset)} samples")
print(f"  Val:   {len(val_loader.dataset)} samples")
print(f"  Test:  {len(test_loader.dataset)} samples")

Using local LFW data: /content/drive/MyDrive/ASD_FaceReg_Modeling_CNN/data/lfw/lfw-deepfunneled/lfw-deepfunneled
Using CSV file: /content/drive/MyDrive/ASD_FaceReg_Modeling_CNN/data/lfw/lfw_allnames.csv
Loading people with >= 30 images...
Found 34 people with >= 30 images from CSV
Selected 34 people from CSV
Note: Using 34 people (requested 50, but only 34 available)
Creating balanced dataset: 34 people x 30 images


KeyboardInterrupt: 

## 4. Experimental Conditions

Three E/I conditions simulate different levels of excitatory-inhibitory balance:

| Condition | alpha | Interpretation |
|-----------|-------|---------------|
| Inhibited | < 1.0 | Inhibition-dominated (I > E) |
| Balanced  | 1.0   | Typical E/I ratio |
| Excited   | > 1.0 | Excitation-dominated (E > I, ASD-like) |


In [ ]:
conditions = [
    {'alpha': 0.5, 'noise': 0.0, 'name': 'Inhibitation'},
    {'alpha': 1, 'noise': 0.0, 'name': 'Balanced'},
    {'alpha': 2, 'noise': 0.0, 'name': 'Excitation'},

]


for i, c in enumerate(conditions, 1):
    print(f"{i}. {c['name']}: slope={c['alpha']}, noise={c['noise']}")


1. Inhibitation: slope=0.5, noise=0.0
2. Balanced: slope=1, noise=0.0
3. Excitation: slope=2, noise=0.0


### 4.1 Inspect Model Architecture

Build a CORnet-Z with the default decoder (`penultimate_dim=0`) to print the full architecture.


In [ ]:
model = build_cornet_for_training(
            num_classes=N_PEOPLE,
            alpha=1,
            noise_std=0,
            freeze_backbone=False,
            penultimate_dim=0,
            penultimate_dropout=0.5,
        )
print(model)

Building CORnet for training: alpha=1, noise=0
Downloading: "https://s3.amazonaws.com/cornet-models/cornet_z-5c427c9c.pth" to /root/.cache/torch/hub/checkpoints/cornet_z-5c427c9c.pth


100%|██████████| 15.8M/15.8M [00:00<00:00, 19.1MB/s]


Decoder rebuilt: 512 -> 100 classes (original architecture)
Training full model
Sequential(
  (V1): CORblock_Z(
    (conv): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
    (nonlin): EIRectifiedLinear()
    (pool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (output): Identity()
  )
  (V2): CORblock_Z(
    (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (nonlin): EIRectifiedLinear()
    (pool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (output): Identity()
  )
  (V4): CORblock_Z(
    (conv): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (nonlin): EIRectifiedLinear()
    (pool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (output): Identity()
  )
  (IT): CORblock_Z(
    (conv): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (nonlin): EIRectifiedLinear()
    (pool): MaxPool2d(kernel

## 5. Reproducibility & Saving Utilities

Fix all random seeds (Python, NumPy, PyTorch, CUDA) and define helper functions for saving JSON, NumPy, and pickle results to Google Drive.


In [ ]:
# ==========================================
# 0. Reproducibility
# ==========================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ==========================================
# 0.1 Saving configuration
# ==========================================
SAVE_BASE = "/content/drive/MyDrive/ASD_FaceReg_Modeling_CNN/results/EIB/cornet"

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def safe_name(name: str) -> str:
    name = name.strip()
    name = re.sub(r"[^a-zA-Z0-9_\-]+", "_", name)
    return name

def save_json(obj, path: str) -> None:
    ensure_dir(os.path.dirname(path))
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def save_numpy(array: np.ndarray, path: str) -> None:
    ensure_dir(os.path.dirname(path))
    np.save(path, array)

def save_npz(matrices_list, path: str) -> None:
    """
    Save list of matrices (possibly multiple runs) into a single .npz.
    Each run is saved as arr_0, arr_1, ...
    """
    ensure_dir(os.path.dirname(path))
    np.savez_compressed(path, *matrices_list)

def save_model_checkpoint(model: torch.nn.Module, ckpt_path: str, metadata: dict) -> None:
    ensure_dir(os.path.dirname(ckpt_path))
    payload = {
        "state_dict": model.state_dict(),
        "metadata": metadata,
    }
    torch.save(payload, ckpt_path)


# ==========================================
# Helper: RSA-only loader (fixed order)
# ==========================================
from torch.utils.data import DataLoader

def unwrap_loader(maybe_wrapped):
    for attr in ["loader", "dataloader", "data_loader", "_loader"]:
        if hasattr(maybe_wrapped, attr):
            inner = getattr(maybe_wrapped, attr)
            return unwrap_loader(inner)
    return maybe_wrapped

def make_rsa_loader(base_loader):
    dl = unwrap_loader(base_loader)

    if not hasattr(dl, "dataset"):
        raise TypeError(f"unwrap no dataset：{type(dl)}")

    return DataLoader(
        dataset=dl.dataset,
        batch_size=dl.batch_size,
        shuffle=False,
        num_workers=getattr(dl, "num_workers", 0),
        pin_memory=getattr(dl, "pin_memory", False),
        drop_last=False,
        persistent_workers=getattr(dl, "persistent_workers", False),
    )




## 6. Main Training Runs

Each run: `build_cornet_for_training` → `train_cornet` → RSA feature extraction → save.
All runs use the VGGFace2 balanced subset with `penultimate_dim=64`, `dropout=0.5`, `lr=1e-4`.


### 6.1 Run A — alpha = [0.5, 1.0, 2.0], 10 runs x 100 epochs

Moderate E/I range. This is one of the two primary experiment batches.


In [ ]:


# ==========================================
# 1. Conditions & Hyperparams
# ==========================================
conditions = [
    {"alpha": 0.5, "noise": 0.0, "name": "Inhibitated"},
    {"alpha": 1.0, "noise": 0.0, "name": "Balanced"},
    {"alpha": 2.0, "noise": 0.0, "name": "Excitated"},
]
N_RUNS = 10
EPOCHS = 100


# ==========================================
# 2. Define loaders (train vs RSA)
# ==========================================
train_loader_train = loaders["train"]
val_loader_train   = loaders["val"]
test_loader_train  = loaders["test"]

# RSA loaders must be deterministic (shuffle=False)
train_loader_rsa = make_rsa_loader(train_loader_train)
test_loader_rsa  = make_rsa_loader(test_loader_train)


# ==========================================
# 3. Storage for Results
# ==========================================
all_raw_histories      = defaultdict(list)
all_raw_matrices_train = defaultdict(list)
all_raw_matrices_test  = defaultdict(list)

# Store labels PER CONDITION (prevents label/matrix misalignment across conditions)
train_labels_cache = {}   # {cond_name: labels_list}
test_labels_cache  = {}   # {cond_name: labels_list}

print(f"Starting {N_RUNS} runs per condition on {DEVICE}...")

from datetime import datetime

EXP_TIME = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

EXP_DIR = os.path.join(
    SAVE_BASE,
    f"{EPOCHS}_{N_RUNS}_{EXP_TIME}"
)
ensure_dir(EXP_DIR)

print(f"Experiment directory: {EXP_DIR}")
# ==========================================
# 4. Run Experiments (Training + RSA)
# ==========================================
for run_idx in tqdm(range(N_RUNS), desc="Total Progress"):

    run_dir = os.path.join(
        EXP_DIR,
        f"run_{run_idx}"
    )
    ensure_dir(run_dir)

    for cond in conditions:
        cond_name = cond["name"]
        cond_dir = os.path.join(run_dir, safe_name(cond_name))
        ensure_dir(cond_dir)

        # -------------------------------
        # A. Initialize Model
        # -------------------------------
        model = build_cornet_for_training(
            num_classes=N_PEOPLE,
            alpha=cond["alpha"],
            noise_std=cond["noise"],
            freeze_backbone=False,
            penultimate_dim=64,
            penultimate_dropout=0.5,
        )

        # -------------------------------
        # B. Train Model
        # -------------------------------
        _, history = train_cornet(
            model,
            train_loader=train_loader_train,
            val_loader=val_loader_train,
            test_loader=test_loader_train,
            epochs=EPOCHS,
            lr=1e-4,
            device=DEVICE,
        )
        all_raw_histories[cond_name].append(history)

        # Save raw training history (per condition, per run)
        save_json(history, os.path.join(cond_dir, f"history_run{run_idx}.json"))

        # -------------------------------
        # C1. RSA on Training Set (RSA loader)
        # -------------------------------
        model.eval()

        matrix_train, label_indices_train = compute_rsa_matrix(
            model,
            train_loader_rsa,
            device=DEVICE,
            layer="penultimate_dense",
            metric="pearson",
        )
        all_raw_matrices_train[cond_name].append(matrix_train)

        labels_train_now = get_person_name_labels(
            train_loader_rsa,
            label_indices_train,
            add_image_number=True,
        )

        if cond_name not in train_labels_cache:
            train_labels_cache[cond_name] = labels_train_now
            print(f"[Train][{cond_name}] Labels example: {labels_train_now[:10]}")
        else:
            assert labels_train_now == train_labels_cache[cond_name], (
                f"[Train][{cond_name}] RSA sample order changed. "
                "Check shuffle/sampler or random augmentations in the RSA dataset."
            )

        # Save raw train RSA outputs (per condition, per run)
        save_numpy(matrix_train, os.path.join(cond_dir, f"rsa_train_run{run_idx}.npy"))
        save_json(labels_train_now, os.path.join(cond_dir, f"rsa_train_labels_run{run_idx}.json"))

        # -------------------------------
        # C2. RSA on Test Set (RSA loader)
        # -------------------------------
        matrix_test, label_indices_test = compute_rsa_matrix(
            model,
            test_loader_rsa,
            device=DEVICE,
            layer="penultimate_dense",
            metric="pearson",
        )
        all_raw_matrices_test[cond_name].append(matrix_test)

        labels_test_now = get_person_name_labels(
            test_loader_rsa,
            label_indices_test,
            add_image_number=True,
        )

        if cond_name not in test_labels_cache:
            test_labels_cache[cond_name] = labels_test_now
            print(f"[Test][{cond_name}] Labels example: {labels_test_now[:10]}")
        else:
            assert labels_test_now == test_labels_cache[cond_name], (
                f"[Test][{cond_name}] RSA sample order changed. "
                "Check shuffle/sampler or random augmentations in the RSA dataset."
            )

        # Save raw test RSA outputs (per condition, per run)
        save_numpy(matrix_test, os.path.join(cond_dir, f"rsa_test_run{run_idx}.npy"))
        save_json(labels_test_now, os.path.join(cond_dir, f"rsa_test_labels_run{run_idx}.json"))

        # -------------------------------
        # Save model checkpoint (per condition, per run)
        # -------------------------------
        ckpt_path = os.path.join(cond_dir, f"model_run{run_idx}.pt")
        ckpt_meta = {
            "seed": SEED,
            "device": str(DEVICE),
            "epochs": EPOCHS,
            "run_idx": run_idx,
            "condition": cond,
            "layer_for_rsa": "penultimate_dense",
        }
        save_model_checkpoint(model, ckpt_path, ckpt_meta)

        # -------------------------------
        # D. Cleanup GPU Memory
        # -------------------------------
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Save run-level raw dict snapshots (useful if you want one file per run)
    # Histories are JSON-serializable; matrices are saved as NPZ.
    run_histories_path = os.path.join(run_dir, "all_raw_histories.json")
    run_mats_train_path = os.path.join(run_dir, "all_raw_matrices_train.npz")
    run_mats_test_path  = os.path.join(run_dir, "all_raw_matrices_test.npz")
    run_labels_train_path = os.path.join(run_dir, "train_labels_by_condition.json")
    run_labels_test_path  = os.path.join(run_dir, "test_labels_by_condition.json")

    save_json(all_raw_histories, run_histories_path)
    save_json(train_labels_cache, run_labels_train_path)
    save_json(test_labels_cache, run_labels_test_path)

    # Save matrices per condition into one NPZ per split
    # Each condition becomes a separate NPZ file inside the run directory.
    for cond in conditions:
        cond_name = cond["name"]
        cond_safe = safe_name(cond_name)

        save_npz(
            all_raw_matrices_train[cond_name],
            os.path.join(run_dir, f"{cond_safe}_train_matrices.npz")
        )
        save_npz(
            all_raw_matrices_test[cond_name],
            os.path.join(run_dir, f"{cond_safe}_test_matrices.npz")
        )

print("All runs completed. Aggregating data...")


Starting 10 runs per condition on cuda...
Experiment directory: /content/drive/MyDrive/ASD_FaceReg_Modeling_CNN/results/EIB/cornet/100_10_2026-01-05_20-02-06


Total Progress:   0%|          | 0/10 [00:00<?, ?it/s]

Building CORnet for training: alpha=0.5, noise=0.0
Decoder rebuilt: 512 -> 64 (ReLU + Dropout) -> 10 classes
Training full model
Starting training for 100 epochs on cuda...


Epoch 1/100:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 1: Train Loss: 2.3086 | Train Acc: 11.50% | Val Loss: 2.3052 | Val Acc: 10.00%


...

Epoch 98: Train Loss: 0.5460 | Train Acc: 78.25% | Val Loss: 0.9651 | Val Acc: 66.00%


Epoch 99/100:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 99: Train Loss: 0.5143 | Train Acc: 80.50% | Val Loss: 0.9387 | Val Acc: 66.00%


Epoch 100/100:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 100: Train Loss: 0.4808 | Train Acc: 83.50% | Val Loss: 0.9345 | Val Acc: 66.00%

Running Final Test Set Evaluation...
FINAL TEST RESULT: Accuracy = 76.00% | Loss = 0.7614
Training complete
Found layer 'ReLU(inplace=True)' in model.
Extracted features: (400, 64)
Found layer 'ReLU(inplace=True)' in model.
Extracted features: (50, 64)
All runs completed. Aggregating data...


#### Run A — Training Curves


In [ ]:
# ==========================================
# 5. Visualization: Training Dynamics
# ==========================================
averaged_histories = {}
metrics = ["train_acc", "val_acc", "train_loss", "val_loss"]

for name, run_list in all_raw_histories.items():
    avg_hist = {}

    for m in metrics:
        vals = [np.array(h.get(m, []), dtype=float) for h in run_list if h.get(m, [])]
        if len(vals) > 0:
            stacked = np.stack(vals, axis=0)  # (n_runs, n_epochs)
            avg_hist[m] = np.mean(stacked, axis=0).tolist()
        else:
            avg_hist[m] = []

    final_test_accs = [h.get("final_test_acc", 0.0) for h in run_list]
    avg_hist["final_test_acc"] = float(np.mean(final_test_accs)) if len(final_test_accs) > 0 else 0.0

    averaged_histories[name] = avg_hist

print("\n" + "=" * 60)
print(f"VISUALIZING AVERAGE TRAINING METRICS ({N_RUNS} Runs)")
print("=" * 60)
visualize_training_comparisons(averaged_histories)


# ==========================================
# 6. Visualization: RSA Matrices (PER CONDITION labels)
# ==========================================
print("\n" + "=" * 60)
print(f"VISUALIZING TRAINING DATA MEAN RSA MATRICES ({N_RUNS} Runs)")
print("=" * 60)
aggregate_and_visualize_rsa(all_raw_matrices_train, train_labels_cache, title_suffix="[Train]")

print("\n" + "=" * 60)
print(f"VISUALIZING TESTING DATA MEAN RSA MATRICES ({N_RUNS} Runs)")
print("=" * 60)
aggregate_and_visualize_rsa(all_raw_matrices_test, test_labels_cache, title_suffix="[Test]")


# ==========================================
# 7. Final Summary Table
# ==========================================
print("\n" + "=" * 80)
print(f"FINAL AVERAGED RESULTS SUMMARY ({N_RUNS} Runs)")
print("=" * 80)
print(f"{'Condition':<30} | {'Train Acc':<12} | {'Val Acc':<12} | {'Test Acc':<12}")
print("-" * 80)

for name, hist in averaged_histories.items():
    train_acc_list = hist.get("train_acc", [])
    val_acc_list   = hist.get("val_acc", [])
    train_acc = train_acc_list[-1] if len(train_acc_list) > 0 else 0.0
    val_acc   = val_acc_list[-1] if len(val_acc_list) > 0 else 0.0
    test_acc  = hist.get("final_test_acc", 0.0)

    print(f"{name:<30} | {train_acc:>10.2f}% | {val_acc:>10.2f}% | {test_acc:>10.2f}%")

print("=" * 80)

# Optional: save final averaged summary to the last run directory (if N_RUNS==1, this is the only run)
# If you want a single global directory for all runs, set a separate path and save there.
if N_RUNS == 1:
    final_dir = os.path.join(SAVE_BASE, f"{EPOCHS}_0")
    save_json(averaged_histories, os.path.join(final_dir, "averaged_histories.json"))


Output hidden; open in https://colab.research.google.com to view.

### 6.2 Run B — alpha = [0.2, 1.0, 5.0], 10 runs x 100 epochs

Wider E/I range (more extreme inhibition and excitation).


In [ ]:


# ==========================================
# 1. Conditions & Hyperparams
# ==========================================
conditions = [
    {"alpha": 0.2, "noise": 0.0, "name": "Inhibitated"},
    {"alpha": 1.0, "noise": 0.0, "name": "Balanced"},
    {"alpha": 5.0, "noise": 0.0, "name": "Excitated"},
]
N_RUNS = 10
EPOCHS = 100


# ==========================================
# 2. Define loaders (train vs RSA)
# ==========================================
train_loader_train = loaders["train"]   # training loader (can be shuffle=True)
val_loader_train   = loaders["val"]
test_loader_train  = loaders["test"]

# RSA loaders must be deterministic (shuffle=False)
train_loader_rsa = make_rsa_loader(train_loader_train)
test_loader_rsa  = make_rsa_loader(test_loader_train)


# ==========================================
# 3. Storage for Results
# ==========================================
all_raw_histories      = defaultdict(list)
all_raw_matrices_train = defaultdict(list)
all_raw_matrices_test  = defaultdict(list)

# Store labels PER CONDITION (prevents label/matrix misalignment across conditions)
train_labels_cache = {}   # {cond_name: labels_list}
test_labels_cache  = {}   # {cond_name: labels_list}

print(f"Starting {N_RUNS} runs per condition on {DEVICE}...")

from datetime import datetime

EXP_TIME = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

EXP_DIR = os.path.join(
    SAVE_BASE,
    f"{EPOCHS}_{N_RUNS}_{EXP_TIME}"
)
ensure_dir(EXP_DIR)

print(f"Experiment directory: {EXP_DIR}")
# ==========================================
# 4. Run Experiments (Training + RSA)
# ==========================================
for run_idx in tqdm(range(N_RUNS), desc="Total Progress"):

    run_dir = os.path.join(
        EXP_DIR,
        f"run_{run_idx}"
    )
    ensure_dir(run_dir)

    for cond in conditions:
        cond_name = cond["name"]
        cond_dir = os.path.join(run_dir, safe_name(cond_name))
        ensure_dir(cond_dir)

        # -------------------------------
        # A. Initialize Model
        # -------------------------------
        model = build_cornet_for_training(
            num_classes=N_PEOPLE,
            alpha=cond["alpha"],
            noise_std=cond["noise"],
            freeze_backbone=False,
            penultimate_dim=64,
            penultimate_dropout=0.5,
        )

        # -------------------------------
        # B. Train Model
        # -------------------------------
        _, history = train_cornet(
            model,
            train_loader=train_loader_train,
            val_loader=val_loader_train,
            test_loader=test_loader_train,
            epochs=EPOCHS,
            lr=1e-4,
            device=DEVICE,
        )
        all_raw_histories[cond_name].append(history)

        # Save raw training history (per condition, per run)
        save_json(history, os.path.join(cond_dir, f"history_run{run_idx}.json"))

        # -------------------------------
        # C1. RSA on Training Set (RSA loader)
        # -------------------------------
        model.eval()

        matrix_train, label_indices_train = compute_rsa_matrix(
            model,
            train_loader_rsa,
            device=DEVICE,
            layer="penultimate_dense",
            metric="pearson",
        )
        all_raw_matrices_train[cond_name].append(matrix_train)

        labels_train_now = get_person_name_labels(
            train_loader_rsa,
            label_indices_train,
            add_image_number=True,
        )

        if cond_name not in train_labels_cache:
            train_labels_cache[cond_name] = labels_train_now
            print(f"[Train][{cond_name}] Labels example: {labels_train_now[:10]}")
        else:
            assert labels_train_now == train_labels_cache[cond_name], (
                f"[Train][{cond_name}] RSA sample order changed. "
                "Check shuffle/sampler or random augmentations in the RSA dataset."
            )

        # Save raw train RSA outputs (per condition, per run)
        save_numpy(matrix_train, os.path.join(cond_dir, f"rsa_train_run{run_idx}.npy"))
        save_json(labels_train_now, os.path.join(cond_dir, f"rsa_train_labels_run{run_idx}.json"))

        # -------------------------------
        # C2. RSA on Test Set (RSA loader)
        # -------------------------------
        matrix_test, label_indices_test = compute_rsa_matrix(
            model,
            test_loader_rsa,
            device=DEVICE,
            layer="penultimate_dense",
            metric="pearson",
        )
        all_raw_matrices_test[cond_name].append(matrix_test)

        labels_test_now = get_person_name_labels(
            test_loader_rsa,
            label_indices_test,
            add_image_number=True,
        )

        if cond_name not in test_labels_cache:
            test_labels_cache[cond_name] = labels_test_now
            print(f"[Test][{cond_name}] Labels example: {labels_test_now[:10]}")
        else:
            assert labels_test_now == test_labels_cache[cond_name], (
                f"[Test][{cond_name}] RSA sample order changed. "
                "Check shuffle/sampler or random augmentations in the RSA dataset."
            )

        # Save raw test RSA outputs (per condition, per run)
        save_numpy(matrix_test, os.path.join(cond_dir, f"rsa_test_run{run_idx}.npy"))
        save_json(labels_test_now, os.path.join(cond_dir, f"rsa_test_labels_run{run_idx}.json"))

        # -------------------------------
        # Save model checkpoint (per condition, per run)
        # -------------------------------
        ckpt_path = os.path.join(cond_dir, f"model_run{run_idx}.pt")
        ckpt_meta = {
            "seed": SEED,
            "device": str(DEVICE),
            "epochs": EPOCHS,
            "run_idx": run_idx,
            "condition": cond,
            "layer_for_rsa": "penultimate_dense",
        }
        save_model_checkpoint(model, ckpt_path, ckpt_meta)

        # -------------------------------
        # D. Cleanup GPU Memory
        # -------------------------------
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Save run-level raw dict snapshots (useful if you want one file per run)
    # Histories are JSON-serializable; matrices are saved as NPZ.
    run_histories_path = os.path.join(run_dir, "all_raw_histories.json")
    run_mats_train_path = os.path.join(run_dir, "all_raw_matrices_train.npz")
    run_mats_test_path  = os.path.join(run_dir, "all_raw_matrices_test.npz")
    run_labels_train_path = os.path.join(run_dir, "train_labels_by_condition.json")
    run_labels_test_path  = os.path.join(run_dir, "test_labels_by_condition.json")

    save_json(all_raw_histories, run_histories_path)
    save_json(train_labels_cache, run_labels_train_path)
    save_json(test_labels_cache, run_labels_test_path)

    # Save matrices per condition into one NPZ per split
    # Each condition becomes a separate NPZ file inside the run directory.
    for cond in conditions:
        cond_name = cond["name"]
        cond_safe = safe_name(cond_name)

        save_npz(
            all_raw_matrices_train[cond_name],
            os.path.join(run_dir, f"{cond_safe}_train_matrices.npz")
        )
        save_npz(
            all_raw_matrices_test[cond_name],
            os.path.join(run_dir, f"{cond_safe}_test_matrices.npz")
        )

print("All runs completed. Aggregating data...")

Starting 10 runs per condition on cuda...
Experiment directory: /content/drive/MyDrive/ASD_FaceReg_Modeling_CNN/results/EIB/cornet/100_10_2026-01-05_22-01-49


Total Progress:   0%|          | 0/10 [00:00<?, ?it/s]

Building CORnet for training: alpha=0.2, noise=0.0
Decoder rebuilt: 512 -> 64 (ReLU + Dropout) -> 10 classes
Training full model
Starting training for 100 epochs on cuda...


Epoch 1/100:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 1: Train Loss: 2.3064 | Train Acc: 10.75% | Val Loss: 2.3059 | Val Acc: 10.00%


...

Epoch 98: Train Loss: 2.2569 | Train Acc: 11.75% | Val Loss: 2.7618 | Val Acc: 10.00%


Epoch 99/100:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 99: Train Loss: 2.2467 | Train Acc: 12.25% | Val Loss: 2.7546 | Val Acc: 10.00%


Epoch 100/100:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 100: Train Loss: 2.2520 | Train Acc: 12.00% | Val Loss: 2.7500 | Val Acc: 10.00%

Running Final Test Set Evaluation...
FINAL TEST RESULT: Accuracy = 10.00% | Loss = 2.5469
Training complete
Found layer 'ReLU(inplace=True)' in model.
Extracted features: (400, 64)
Found layer 'ReLU(inplace=True)' in model.
Extracted features: (50, 64)
All runs completed. Aggregating data...


#### Run B — Training Curves


In [ ]:
# ==========================================
# 5. Visualization: Training Dynamics
# ==========================================
averaged_histories = {}
metrics = ["train_acc", "val_acc", "train_loss", "val_loss"]

for name, run_list in all_raw_histories.items():
    avg_hist = {}

    for m in metrics:
        vals = [np.array(h.get(m, []), dtype=float) for h in run_list if h.get(m, [])]
        if len(vals) > 0:
            stacked = np.stack(vals, axis=0)  # (n_runs, n_epochs)
            avg_hist[m] = np.mean(stacked, axis=0).tolist()
        else:
            avg_hist[m] = []

    final_test_accs = [h.get("final_test_acc", 0.0) for h in run_list]
    avg_hist["final_test_acc"] = float(np.mean(final_test_accs)) if len(final_test_accs) > 0 else 0.0

    averaged_histories[name] = avg_hist

print("\n" + "=" * 60)
print(f"VISUALIZING AVERAGE TRAINING METRICS ({N_RUNS} Runs)")
print("=" * 60)
visualize_training_comparisons(averaged_histories)


# ==========================================
# 6. Visualization: RSA Matrices (PER CONDITION labels)
# ==========================================
print("\n" + "=" * 60)
print(f"VISUALIZING TRAINING DATA MEAN RSA MATRICES ({N_RUNS} Runs)")
print("=" * 60)
aggregate_and_visualize_rsa(all_raw_matrices_train, train_labels_cache, title_suffix="[Train]")

print("\n" + "=" * 60)
print(f"VISUALIZING TESTING DATA MEAN RSA MATRICES ({N_RUNS} Runs)")
print("=" * 60)
aggregate_and_visualize_rsa(all_raw_matrices_test, test_labels_cache, title_suffix="[Test]")


# ==========================================
# 7. Final Summary Table
# ==========================================
print("\n" + "=" * 80)
print(f"FINAL AVERAGED RESULTS SUMMARY ({N_RUNS} Runs)")
print("=" * 80)
print(f"{'Condition':<30} | {'Train Acc':<12} | {'Val Acc':<12} | {'Test Acc':<12}")
print("-" * 80)

for name, hist in averaged_histories.items():
    train_acc_list = hist.get("train_acc", [])
    val_acc_list   = hist.get("val_acc", [])
    train_acc = train_acc_list[-1] if len(train_acc_list) > 0 else 0.0
    val_acc   = val_acc_list[-1] if len(val_acc_list) > 0 else 0.0
    test_acc  = hist.get("final_test_acc", 0.0)

    print(f"{name:<30} | {train_acc:>10.2f}% | {val_acc:>10.2f}% | {test_acc:>10.2f}%")

print("=" * 80)

# Optional: save final averaged summary to the last run directory (if N_RUNS==1, this is the only run)
# If you want a single global directory for all runs, set a separate path and save there.
if N_RUNS == 1:
    final_dir = os.path.join(SAVE_BASE, f"{EPOCHS}_0")
    save_json(averaged_histories, os.path.join(final_dir, "averaged_histories.json"))

Output hidden; open in https://colab.research.google.com to view.

## 7. Visualization & Export

Aggregate training curves across runs and conditions, then save all results.


### 7.1 Averaged Training Dynamics


In [ ]:
# ==========================================
# 5. Visualization: Training Dynamics
# ==========================================
averaged_histories = {}
metrics = ["train_acc", "val_acc", "train_loss", "val_loss"]

for name, run_list in all_raw_histories.items():
    avg_hist = {}

    for m in metrics:
        vals = [np.array(h.get(m, []), dtype=float) for h in run_list if h.get(m, [])]
        if len(vals) > 0:
            stacked = np.stack(vals, axis=0)  # (n_runs, n_epochs)
            avg_hist[m] = np.mean(stacked, axis=0).tolist()
        else:
            avg_hist[m] = []

    final_test_accs = [h.get("final_test_acc", 0.0) for h in run_list]
    avg_hist["final_test_acc"] = float(np.mean(final_test_accs)) if len(final_test_accs) > 0 else 0.0

    averaged_histories[name] = avg_hist

print("\n" + "=" * 60)
print(f"VISUALIZING AVERAGE TRAINING METRICS ({N_RUNS} Runs)")
print("=" * 60)
visualize_training_comparisons(averaged_histories)


# ==========================================
# 6. Visualization: RSA Matrices (PER CONDITION labels)
# ==========================================
print("\n" + "=" * 60)
print(f"VISUALIZING TRAINING DATA MEAN RSA MATRICES ({N_RUNS} Runs)")
print("=" * 60)
aggregate_and_visualize_rsa(all_raw_matrices_train, train_labels_cache, title_suffix="[Train]")

print("\n" + "=" * 60)
print(f"VISUALIZING TESTING DATA MEAN RSA MATRICES ({N_RUNS} Runs)")
print("=" * 60)
aggregate_and_visualize_rsa(all_raw_matrices_test, test_labels_cache, title_suffix="[Test]")


# ==========================================
# 7. Final Summary Table
# ==========================================
print("\n" + "=" * 80)
print(f"FINAL AVERAGED RESULTS SUMMARY ({N_RUNS} Runs)")
print("=" * 80)
print(f"{'Condition':<30} | {'Train Acc':<12} | {'Val Acc':<12} | {'Test Acc':<12}")
print("-" * 80)

for name, hist in averaged_histories.items():
    train_acc_list = hist.get("train_acc", [])
    val_acc_list   = hist.get("val_acc", [])
    train_acc = train_acc_list[-1] if len(train_acc_list) > 0 else 0.0
    val_acc   = val_acc_list[-1] if len(val_acc_list) > 0 else 0.0
    test_acc  = hist.get("final_test_acc", 0.0)

    print(f"{name:<30} | {train_acc:>10.2f}% | {val_acc:>10.2f}% | {test_acc:>10.2f}%")

print("=" * 80)

# Optional: save final averaged summary to the last run directory (if N_RUNS==1, this is the only run)
# If you want a single global directory for all runs, set a separate path and save there.
if N_RUNS == 1:
    final_dir = os.path.join(SAVE_BASE, f"{EPOCHS}_0")
    save_json(averaged_histories, os.path.join(final_dir, "averaged_histories.json"))


Output hidden; open in https://colab.research.google.com to view.

### 7.2 Save Results (Pickle)


In [ ]:
import pickle

objects_to_save = {
    "averaged_histories": averaged_histories,
    "all_raw_histories": all_raw_histories
}

save_paths = [
    "/content/drive/MyDrive/ASD_FaceReg_Modeling_CNN/results/EIB/cornet/50_10/averaged_histories.pkl",
    "/content/drive/MyDrive/ASD_FaceReg_Modeling_CNN/results/EIB/cornet/50_10/all_raw_histories.pkl"
]

for (name, obj), path in zip(objects_to_save.items(), save_paths):
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"Saved {name} to {path}")



Saved averaged_histories to /content/drive/MyDrive/ASD_FaceReg_Modeling_CNN/results/EIB/cornet/50_10/averaged_histories.pkl
Saved all_raw_histories to /content/drive/MyDrive/ASD_FaceReg_Modeling_CNN/results/EIB/cornet/50_10/all_raw_histories.pkl


### 7.3 Colab Session Cleanup


In [ ]:
import time
from google.colab import runtime

print("Finished! Will disconnect in 10 seconds...")
time.sleep(10)
runtime.unassign()


Finished! Will disconnect in 10 seconds...


## 8. Pilot & Parameter-Search Runs

Exploratory single-run experiments to test different dataset sizes, batch sizes, decoder widths, and dropout rates before committing to a full 10-run batch.

---


### 8.1 Extended Dataset Pilot — 150 identities, alpha=[0.5, 1.0, 2.0], 1 run x 100 epochs

`penultimate_dim=64`, `dropout=0.5`


In [ ]:


# ==========================================
# 1. Conditions & Hyperparams
# ==========================================
conditions = [
    {"alpha": 0.5, "noise": 0.0, "name": "Inhibitated"},
    {"alpha": 1.0, "noise": 0.0, "name": "Balanced"},
    {"alpha": 2.0, "noise": 0.0, "name": "Excitated"},
]
N_RUNS = 1
EPOCHS = 100


# ==========================================
# 2. Define loaders (train vs RSA)
# ==========================================
train_loader_train = loaders["train"]
val_loader_train   = loaders["val"]
test_loader_train  = loaders["test"]

# RSA loaders must be deterministic (shuffle=False)
train_loader_rsa = make_rsa_loader(train_loader_train)
test_loader_rsa  = make_rsa_loader(test_loader_train)


# ==========================================
# 3. Storage for Results
# ==========================================
all_raw_histories      = defaultdict(list)
all_raw_matrices_train = defaultdict(list)
all_raw_matrices_test  = defaultdict(list)

# Store labels PER CONDITION (prevents label/matrix misalignment across conditions)
train_labels_cache = {}   # {cond_name: labels_list}
test_labels_cache  = {}   # {cond_name: labels_list}

print(f"Starting {N_RUNS} runs per condition on {DEVICE}...")

from datetime import datetime

EXP_TIME = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

EXP_DIR = os.path.join(
    SAVE_BASE,
    f"{EPOCHS}_{N_RUNS}_{str(N_PEOPLE)}_{str(IMGS_PER_PERSON)}_{EXP_TIME}"
)
ensure_dir(EXP_DIR)

print(f"Experiment directory: {EXP_DIR}")
# ==========================================
# 4. Run Experiments (Training + RSA)
# ==========================================
for run_idx in tqdm(range(N_RUNS), desc="Total Progress"):

    run_dir = os.path.join(
        EXP_DIR,
        f"run_{run_idx}"
    )
    ensure_dir(run_dir)

    for cond in conditions:
        cond_name = cond["name"]
        cond_dir = os.path.join(run_dir, safe_name(cond_name))
        ensure_dir(cond_dir)

        # -------------------------------
        # A. Initialize Model
        # -------------------------------
        model = build_cornet_for_training(
            num_classes=N_PEOPLE,
            alpha=cond["alpha"],
            noise_std=cond["noise"],
            freeze_backbone=False,
            penultimate_dim=64,
            penultimate_dropout=0.5,
        )

        # -------------------------------
        # B. Train Model
        # -------------------------------
        _, history = train_cornet(
            model,
            train_loader=train_loader_train,
            val_loader=val_loader_train,
            test_loader=test_loader_train,
            epochs=EPOCHS,
            lr=1e-4,
            device=DEVICE,
        )
        all_raw_histories[cond_name].append(history)

        # Save raw training history (per condition, per run)
        save_json(history, os.path.join(cond_dir, f"history_run{run_idx}.json"))

        # -------------------------------
        # C1. RSA on Training Set (RSA loader)
        # -------------------------------
        model.eval()

        matrix_train, label_indices_train = compute_rsa_matrix(
            model,
            train_loader_rsa,
            device=DEVICE,
            layer="penultimate_dense",
            metric="pearson",
        )
        all_raw_matrices_train[cond_name].append(matrix_train)

        labels_train_now = get_person_name_labels(
            train_loader_rsa,
            label_indices_train,
            add_image_number=True,
        )

        if cond_name not in train_labels_cache:
            train_labels_cache[cond_name] = labels_train_now
            print(f"[Train][{cond_name}] Labels example: {labels_train_now[:10]}")
        else:
            assert labels_train_now == train_labels_cache[cond_name], (
                f"[Train][{cond_name}] RSA sample order changed. "
                "Check shuffle/sampler or random augmentations in the RSA dataset."
            )

        # Save raw train RSA outputs (per condition, per run)
        save_numpy(matrix_train, os.path.join(cond_dir, f"rsa_train_run{run_idx}.npy"))
        save_json(labels_train_now, os.path.join(cond_dir, f"rsa_train_labels_run{run_idx}.json"))

        # -------------------------------
        # C2. RSA on Test Set (RSA loader)
        # -------------------------------
        matrix_test, label_indices_test = compute_rsa_matrix(
            model,
            test_loader_rsa,
            device=DEVICE,
            layer="penultimate_dense",
            metric="pearson",
        )
        all_raw_matrices_test[cond_name].append(matrix_test)

        labels_test_now = get_person_name_labels(
            test_loader_rsa,
            label_indices_test,
            add_image_number=True,
        )

        if cond_name not in test_labels_cache:
            test_labels_cache[cond_name] = labels_test_now
            print(f"[Test][{cond_name}] Labels example: {labels_test_now[:10]}")
        else:
            assert labels_test_now == test_labels_cache[cond_name], (
                f"[Test][{cond_name}] RSA sample order changed. "
                "Check shuffle/sampler or random augmentations in the RSA dataset."
            )

        # Save raw test RSA outputs (per condition, per run)
        save_numpy(matrix_test, os.path.join(cond_dir, f"rsa_test_run{run_idx}.npy"))
        save_json(labels_test_now, os.path.join(cond_dir, f"rsa_test_labels_run{run_idx}.json"))

        # -------------------------------
        # Save model checkpoint (per condition, per run)
        # -------------------------------
        ckpt_path = os.path.join(cond_dir, f"model_run{run_idx}.pt")
        ckpt_meta = {
            "seed": SEED,
            "device": str(DEVICE),
            "epochs": EPOCHS,
            "run_idx": run_idx,
            "condition": cond,
            "layer_for_rsa": "penultimate_dense",
        }
        save_model_checkpoint(model, ckpt_path, ckpt_meta)

        # -------------------------------
        # D. Cleanup GPU Memory
        # -------------------------------
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Save run-level raw dict snapshots (useful if you want one file per run)
    # Histories are JSON-serializable; matrices are saved as NPZ.
    run_histories_path = os.path.join(run_dir, "all_raw_histories.json")
    run_mats_train_path = os.path.join(run_dir, "all_raw_matrices_train.npz")
    run_mats_test_path  = os.path.join(run_dir, "all_raw_matrices_test.npz")
    run_labels_train_path = os.path.join(run_dir, "train_labels_by_condition.json")
    run_labels_test_path  = os.path.join(run_dir, "test_labels_by_condition.json")

    save_json(all_raw_histories, run_histories_path)
    save_json(train_labels_cache, run_labels_train_path)
    save_json(test_labels_cache, run_labels_test_path)

    # Save matrices per condition into one NPZ per split
    # Each condition becomes a separate NPZ file inside the run directory.
    for cond in conditions:
        cond_name = cond["name"]
        cond_safe = safe_name(cond_name)

        save_npz(
            all_raw_matrices_train[cond_name],
            os.path.join(run_dir, f"{cond_safe}_train_matrices.npz")
        )
        save_npz(
            all_raw_matrices_test[cond_name],
            os.path.join(run_dir, f"{cond_safe}_test_matrices.npz")
        )

print("All runs completed. Aggregating data...")

# ==========================================
# 5. Visualization: Training Dynamics
# ==========================================
averaged_histories = {}
metrics = ["train_acc", "val_acc", "train_loss", "val_loss"]

for name, run_list in all_raw_histories.items():
    avg_hist = {}

    for m in metrics:
        vals = [np.array(h.get(m, []), dtype=float) for h in run_list if h.get(m, [])]
        if len(vals) > 0:
            stacked = np.stack(vals, axis=0)  # (n_runs, n_epochs)
            avg_hist[m] = np.mean(stacked, axis=0).tolist()
        else:
            avg_hist[m] = []

    final_test_accs = [h.get("final_test_acc", 0.0) for h in run_list]
    avg_hist["final_test_acc"] = float(np.mean(final_test_accs)) if len(final_test_accs) > 0 else 0.0

    averaged_histories[name] = avg_hist

print("\n" + "=" * 60)
print(f"VISUALIZING AVERAGE TRAINING METRICS ({N_RUNS} Runs)")
print("=" * 60)
visualize_training_comparisons(averaged_histories)


# ==========================================
# 6. Visualization: RSA Matrices (PER CONDITION labels)
# ==========================================
print("\n" + "=" * 60)
print(f"VISUALIZING TRAINING DATA MEAN RSA MATRICES ({N_RUNS} Runs)")
print("=" * 60)
aggregate_and_visualize_rsa(all_raw_matrices_train, train_labels_cache, title_suffix="[Train]")

print("\n" + "=" * 60)
print(f"VISUALIZING TESTING DATA MEAN RSA MATRICES ({N_RUNS} Runs)")
print("=" * 60)
aggregate_and_visualize_rsa(all_raw_matrices_test, test_labels_cache, title_suffix="[Test]")


# ==========================================
# 7. Final Summary Table
# ==========================================
print("\n" + "=" * 80)
print(f"FINAL AVERAGED RESULTS SUMMARY ({N_RUNS} Runs)")
print("=" * 80)
print(f"{'Condition':<30} | {'Train Acc':<12} | {'Val Acc':<12} | {'Test Acc':<12}")
print("-" * 80)

for name, hist in averaged_histories.items():
    train_acc_list = hist.get("train_acc", [])
    val_acc_list   = hist.get("val_acc", [])
    train_acc = train_acc_list[-1] if len(train_acc_list) > 0 else 0.0
    val_acc   = val_acc_list[-1] if len(val_acc_list) > 0 else 0.0
    test_acc  = hist.get("final_test_acc", 0.0)

    print(f"{name:<30} | {train_acc:>10.2f}% | {val_acc:>10.2f}% | {test_acc:>10.2f}%")

print("=" * 80)

# Optional: save final averaged summary to the last run directory (if N_RUNS==1, this is the only run)
# If you want a single global directory for all runs, set a separate path and save there.
if N_RUNS == 1:
    final_dir = os.path.join(SAVE_BASE, f"{EPOCHS}_0")
    save_json(averaged_histories, os.path.join(final_dir, "averaged_histories.json"))

Output hidden; open in https://colab.research.google.com to view.

### 8.2 Large Batch Pilot — batch_size=512, 1 run x 200 epochs

Reloads data with `batch_size=512`, then trains with `penultimate_dim=64`, `dropout=0.5`.


In [ ]:
BATCH_SIZE=512

loaders = get_split_dataloaders(
    data_dir=BALANCED_DATA_DIR,
    batch_size=BATCH_SIZE,
    split_ratio=(0.8, 0.1, 0.1),
    num_workers=0,
    seed=42
)

train_loader = loaders['train']
val_loader = loaders['val']
test_loader = loaders['test']

print(f"\nData loading complete!")
print(f"  Train: {len(train_loader.dataset)} samples")
print(f"  Val:   {len(val_loader.dataset)} samples")
print(f"  Test:  {len(test_loader.dataset)} samples")
# ==========================================
# 1. Conditions & Hyperparams
# ==========================================
conditions = [
    {"alpha": 0.5, "noise": 0.0, "name": "Inhibitated"},
    {"alpha": 1.0, "noise": 0.0, "name": "Balanced"},
    {"alpha": 2.0, "noise": 0.0, "name": "Excitated"},
]
N_RUNS = 1
EPOCHS = 200


# ==========================================
# 2. Define loaders (train vs RSA)
# ==========================================
train_loader_train = loaders["train"]
val_loader_train   = loaders["val"]
test_loader_train  = loaders["test"]

# RSA loaders must be deterministic (shuffle=False)
train_loader_rsa = make_rsa_loader(train_loader_train)
test_loader_rsa  = make_rsa_loader(test_loader_train)


# ==========================================
# 3. Storage for Results
# ==========================================
all_raw_histories      = defaultdict(list)
all_raw_matrices_train = defaultdict(list)
all_raw_matrices_test  = defaultdict(list)

# Store labels PER CONDITION (prevents label/matrix misalignment across conditions)
train_labels_cache = {}   # {cond_name: labels_list}
test_labels_cache  = {}   # {cond_name: labels_list}

print(f"Starting {N_RUNS} runs per condition on {DEVICE}...")

from datetime import datetime

EXP_TIME = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

EXP_DIR = os.path.join(
    SAVE_BASE,
    f"{EPOCHS}_{N_RUNS}_{str(N_PEOPLE)}_{str(IMGS_PER_PERSON)}_{EXP_TIME}"
)
ensure_dir(EXP_DIR)

print(f"Experiment directory: {EXP_DIR}")
# ==========================================
# 4. Run Experiments (Training + RSA)
# ==========================================
for run_idx in tqdm(range(N_RUNS), desc="Total Progress"):

    run_dir = os.path.join(
        EXP_DIR,
        f"run_{run_idx}"
    )
    ensure_dir(run_dir)

    for cond in conditions:
        cond_name = cond["name"]
        cond_dir = os.path.join(run_dir, safe_name(cond_name))
        ensure_dir(cond_dir)

        # -------------------------------
        # A. Initialize Model
        # -------------------------------
        model = build_cornet_for_training(
            num_classes=N_PEOPLE,
            alpha=cond["alpha"],
            noise_std=cond["noise"],
            freeze_backbone=False,
            penultimate_dim=64,
            penultimate_dropout=0.5,
        )

        # -------------------------------
        # B. Train Model
        # -------------------------------
        _, history = train_cornet(
            model,
            train_loader=train_loader_train,
            val_loader=val_loader_train,
            test_loader=test_loader_train,
            epochs=EPOCHS,
            lr=1e-4,
            device=DEVICE,
        )
        all_raw_histories[cond_name].append(history)

        # Save raw training history (per condition, per run)
        save_json(history, os.path.join(cond_dir, f"history_run{run_idx}.json"))

        # -------------------------------
        # C1. RSA on Training Set (RSA loader)
        # -------------------------------
        model.eval()

        matrix_train, label_indices_train = compute_rsa_matrix(
            model,
            train_loader_rsa,
            device=DEVICE,
            layer="penultimate_dense",
            metric="pearson",
        )
        all_raw_matrices_train[cond_name].append(matrix_train)

        labels_train_now = get_person_name_labels(
            train_loader_rsa,
            label_indices_train,
            add_image_number=True,
        )

        if cond_name not in train_labels_cache:
            train_labels_cache[cond_name] = labels_train_now
            print(f"[Train][{cond_name}] Labels example: {labels_train_now[:10]}")
        else:
            assert labels_train_now == train_labels_cache[cond_name], (
                f"[Train][{cond_name}] RSA sample order changed. "
                "Check shuffle/sampler or random augmentations in the RSA dataset."
            )

        # Save raw train RSA outputs (per condition, per run)
        save_numpy(matrix_train, os.path.join(cond_dir, f"rsa_train_run{run_idx}.npy"))
        save_json(labels_train_now, os.path.join(cond_dir, f"rsa_train_labels_run{run_idx}.json"))

        # -------------------------------
        # C2. RSA on Test Set (RSA loader)
        # -------------------------------
        matrix_test, label_indices_test = compute_rsa_matrix(
            model,
            test_loader_rsa,
            device=DEVICE,
            layer="penultimate_dense",
            metric="pearson",
        )
        all_raw_matrices_test[cond_name].append(matrix_test)

        labels_test_now = get_person_name_labels(
            test_loader_rsa,
            label_indices_test,
            add_image_number=True,
        )

        if cond_name not in test_labels_cache:
            test_labels_cache[cond_name] = labels_test_now
            print(f"[Test][{cond_name}] Labels example: {labels_test_now[:10]}")
        else:
            assert labels_test_now == test_labels_cache[cond_name], (
                f"[Test][{cond_name}] RSA sample order changed. "
                "Check shuffle/sampler or random augmentations in the RSA dataset."
            )

        # Save raw test RSA outputs (per condition, per run)
        save_numpy(matrix_test, os.path.join(cond_dir, f"rsa_test_run{run_idx}.npy"))
        save_json(labels_test_now, os.path.join(cond_dir, f"rsa_test_labels_run{run_idx}.json"))

        # -------------------------------
        # Save model checkpoint (per condition, per run)
        # -------------------------------
        ckpt_path = os.path.join(cond_dir, f"model_run{run_idx}.pt")
        ckpt_meta = {
            "seed": SEED,
            "device": str(DEVICE),
            "epochs": EPOCHS,
            "run_idx": run_idx,
            "condition": cond,
            "layer_for_rsa": "penultimate_dense",
        }
        save_model_checkpoint(model, ckpt_path, ckpt_meta)

        # -------------------------------
        # D. Cleanup GPU Memory
        # -------------------------------
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Save run-level raw dict snapshots (useful if you want one file per run)
    # Histories are JSON-serializable; matrices are saved as NPZ.
    run_histories_path = os.path.join(run_dir, "all_raw_histories.json")
    run_mats_train_path = os.path.join(run_dir, "all_raw_matrices_train.npz")
    run_mats_test_path  = os.path.join(run_dir, "all_raw_matrices_test.npz")
    run_labels_train_path = os.path.join(run_dir, "train_labels_by_condition.json")
    run_labels_test_path  = os.path.join(run_dir, "test_labels_by_condition.json")

    save_json(all_raw_histories, run_histories_path)
    save_json(train_labels_cache, run_labels_train_path)
    save_json(test_labels_cache, run_labels_test_path)

    # Save matrices per condition into one NPZ per split
    # Each condition becomes a separate NPZ file inside the run directory.
    for cond in conditions:
        cond_name = cond["name"]
        cond_safe = safe_name(cond_name)

        save_npz(
            all_raw_matrices_train[cond_name],
            os.path.join(run_dir, f"{cond_safe}_train_matrices.npz")
        )
        save_npz(
            all_raw_matrices_test[cond_name],
            os.path.join(run_dir, f"{cond_safe}_test_matrices.npz")
        )

print("All runs completed. Aggregating data...")

# ==========================================
# 5. Visualization: Training Dynamics
# ==========================================
averaged_histories = {}
metrics = ["train_acc", "val_acc", "train_loss", "val_loss"]

for name, run_list in all_raw_histories.items():
    avg_hist = {}

    for m in metrics:
        vals = [np.array(h.get(m, []), dtype=float) for h in run_list if h.get(m, [])]
        if len(vals) > 0:
            stacked = np.stack(vals, axis=0)  # (n_runs, n_epochs)
            avg_hist[m] = np.mean(stacked, axis=0).tolist()
        else:
            avg_hist[m] = []

    final_test_accs = [h.get("final_test_acc", 0.0) for h in run_list]
    avg_hist["final_test_acc"] = float(np.mean(final_test_accs)) if len(final_test_accs) > 0 else 0.0

    averaged_histories[name] = avg_hist

print("\n" + "=" * 60)
print(f"VISUALIZING AVERAGE TRAINING METRICS ({N_RUNS} Runs)")
print("=" * 60)
visualize_training_comparisons(averaged_histories)


# ==========================================
# 6. Visualization: RSA Matrices (PER CONDITION labels)
# ==========================================
print("\n" + "=" * 60)
print(f"VISUALIZING TRAINING DATA MEAN RSA MATRICES ({N_RUNS} Runs)")
print("=" * 60)
aggregate_and_visualize_rsa(all_raw_matrices_train, train_labels_cache, title_suffix="[Train]")

print("\n" + "=" * 60)
print(f"VISUALIZING TESTING DATA MEAN RSA MATRICES ({N_RUNS} Runs)")
print("=" * 60)
aggregate_and_visualize_rsa(all_raw_matrices_test, test_labels_cache, title_suffix="[Test]")


# ==========================================
# 7. Final Summary Table
# ==========================================
print("\n" + "=" * 80)
print(f"FINAL AVERAGED RESULTS SUMMARY ({N_RUNS} Runs)")
print("=" * 80)
print(f"{'Condition':<30} | {'Train Acc':<12} | {'Val Acc':<12} | {'Test Acc':<12}")
print("-" * 80)

for name, hist in averaged_histories.items():
    train_acc_list = hist.get("train_acc", [])
    val_acc_list   = hist.get("val_acc", [])
    train_acc = train_acc_list[-1] if len(train_acc_list) > 0 else 0.0
    val_acc   = val_acc_list[-1] if len(val_acc_list) > 0 else 0.0
    test_acc  = hist.get("final_test_acc", 0.0)

    print(f"{name:<30} | {train_acc:>10.2f}% | {val_acc:>10.2f}% | {test_acc:>10.2f}%")

print("=" * 80)

# Optional: save final averaged summary to the last run directory (if N_RUNS==1, this is the only run)
# If you want a single global directory for all runs, set a separate path and save there.
if N_RUNS == 1:
    final_dir = os.path.join(SAVE_BASE, f"{EPOCHS}_0")
    save_json(averaged_histories, os.path.join(final_dir, "averaged_histories.json"))

Output hidden; open in https://colab.research.google.com to view.

### 8.3 Large Batch Continued — batch_size=512, 1 run x 200 epochs

Same config as 8.2, re-run with existing loaders. `penultimate_dim=64`, `dropout=0.5`.


In [ ]:

# ==========================================
# 1. Conditions & Hyperparams
# ==========================================
conditions = [
    {"alpha": 0.5, "noise": 0.0, "name": "Inhibitated"},
    {"alpha": 1.0, "noise": 0.0, "name": "Balanced"},
    {"alpha": 2.0, "noise": 0.0, "name": "Excitated"},
]
N_RUNS = 1
EPOCHS = 200


# ==========================================
# 2. Define loaders (train vs RSA)
# ==========================================
train_loader_train = loaders["train"]
val_loader_train   = loaders["val"]
test_loader_train  = loaders["test"]

# RSA loaders must be deterministic (shuffle=False)
train_loader_rsa = make_rsa_loader(train_loader_train)
test_loader_rsa  = make_rsa_loader(test_loader_train)


# ==========================================
# 3. Storage for Results
# ==========================================
all_raw_histories      = defaultdict(list)
all_raw_matrices_train = defaultdict(list)
all_raw_matrices_test  = defaultdict(list)

# Store labels PER CONDITION (prevents label/matrix misalignment across conditions)
train_labels_cache = {}   # {cond_name: labels_list}
test_labels_cache  = {}   # {cond_name: labels_list}

print(f"Starting {N_RUNS} runs per condition on {DEVICE}...")

from datetime import datetime

EXP_TIME = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

EXP_DIR = os.path.join(
    SAVE_BASE,
    f"{EPOCHS}_{N_RUNS}_{str(N_PEOPLE)}_{str(IMGS_PER_PERSON)}_{EXP_TIME}"
)
ensure_dir(EXP_DIR)

print(f"Experiment directory: {EXP_DIR}")
# ==========================================
# 4. Run Experiments (Training + RSA)
# ==========================================
for run_idx in tqdm(range(N_RUNS), desc="Total Progress"):

    run_dir = os.path.join(
        EXP_DIR,
        f"run_{run_idx}"
    )
    ensure_dir(run_dir)

    for cond in conditions:
        cond_name = cond["name"]
        cond_dir = os.path.join(run_dir, safe_name(cond_name))
        ensure_dir(cond_dir)

        # -------------------------------
        # A. Initialize Model
        # -------------------------------
        model = build_cornet_for_training(
            num_classes=N_PEOPLE,
            alpha=cond["alpha"],
            noise_std=cond["noise"],
            freeze_backbone=False,
            penultimate_dim=64,
            penultimate_dropout=0.5,
        )

        # -------------------------------
        # B. Train Model
        # -------------------------------
        _, history = train_cornet(
            model,
            train_loader=train_loader_train,
            val_loader=val_loader_train,
            test_loader=test_loader_train,
            epochs=EPOCHS,
            lr=1e-4,
            device=DEVICE,
        )
        all_raw_histories[cond_name].append(history)

        # Save raw training history (per condition, per run)
        save_json(history, os.path.join(cond_dir, f"history_run{run_idx}.json"))

        # -------------------------------
        # C1. RSA on Training Set (RSA loader)
        # -------------------------------
        model.eval()

        matrix_train, label_indices_train = compute_rsa_matrix(
            model,
            train_loader_rsa,
            device=DEVICE,
            layer="penultimate_dense",
            metric="pearson",
        )
        all_raw_matrices_train[cond_name].append(matrix_train)

        labels_train_now = get_person_name_labels(
            train_loader_rsa,
            label_indices_train,
            add_image_number=True,
        )

        if cond_name not in train_labels_cache:
            train_labels_cache[cond_name] = labels_train_now
            print(f"[Train][{cond_name}] Labels example: {labels_train_now[:10]}")
        else:
            assert labels_train_now == train_labels_cache[cond_name], (
                f"[Train][{cond_name}] RSA sample order changed. "
                "Check shuffle/sampler or random augmentations in the RSA dataset."
            )

        # Save raw train RSA outputs (per condition, per run)
        save_numpy(matrix_train, os.path.join(cond_dir, f"rsa_train_run{run_idx}.npy"))
        save_json(labels_train_now, os.path.join(cond_dir, f"rsa_train_labels_run{run_idx}.json"))

        # -------------------------------
        # C2. RSA on Test Set (RSA loader)
        # -------------------------------
        matrix_test, label_indices_test = compute_rsa_matrix(
            model,
            test_loader_rsa,
            device=DEVICE,
            layer="penultimate_dense",
            metric="pearson",
        )
        all_raw_matrices_test[cond_name].append(matrix_test)

        labels_test_now = get_person_name_labels(
            test_loader_rsa,
            label_indices_test,
            add_image_number=True,
        )

        if cond_name not in test_labels_cache:
            test_labels_cache[cond_name] = labels_test_now
            print(f"[Test][{cond_name}] Labels example: {labels_test_now[:10]}")
        else:
            assert labels_test_now == test_labels_cache[cond_name], (
                f"[Test][{cond_name}] RSA sample order changed. "
                "Check shuffle/sampler or random augmentations in the RSA dataset."
            )

        # Save raw test RSA outputs (per condition, per run)
        save_numpy(matrix_test, os.path.join(cond_dir, f"rsa_test_run{run_idx}.npy"))
        save_json(labels_test_now, os.path.join(cond_dir, f"rsa_test_labels_run{run_idx}.json"))

        # -------------------------------
        # Save model checkpoint (per condition, per run)
        # -------------------------------
        ckpt_path = os.path.join(cond_dir, f"model_run{run_idx}.pt")
        ckpt_meta = {
            "seed": SEED,
            "device": str(DEVICE),
            "epochs": EPOCHS,
            "run_idx": run_idx,
            "condition": cond,
            "layer_for_rsa": "penultimate_dense",
        }
        save_model_checkpoint(model, ckpt_path, ckpt_meta)

        # -------------------------------
        # D. Cleanup GPU Memory
        # -------------------------------
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Save run-level raw dict snapshots (useful if you want one file per run)
    # Histories are JSON-serializable; matrices are saved as NPZ.
    run_histories_path = os.path.join(run_dir, "all_raw_histories.json")
    run_mats_train_path = os.path.join(run_dir, "all_raw_matrices_train.npz")
    run_mats_test_path  = os.path.join(run_dir, "all_raw_matrices_test.npz")
    run_labels_train_path = os.path.join(run_dir, "train_labels_by_condition.json")
    run_labels_test_path  = os.path.join(run_dir, "test_labels_by_condition.json")

    save_json(all_raw_histories, run_histories_path)
    save_json(train_labels_cache, run_labels_train_path)
    save_json(test_labels_cache, run_labels_test_path)

    # Save matrices per condition into one NPZ per split
    # Each condition becomes a separate NPZ file inside the run directory.
    for cond in conditions:
        cond_name = cond["name"]
        cond_safe = safe_name(cond_name)

        save_npz(
            all_raw_matrices_train[cond_name],
            os.path.join(run_dir, f"{cond_safe}_train_matrices.npz")
        )
        save_npz(
            all_raw_matrices_test[cond_name],
            os.path.join(run_dir, f"{cond_safe}_test_matrices.npz")
        )

print("All runs completed. Aggregating data...")

# ==========================================
# 5. Visualization: Training Dynamics
# ==========================================
averaged_histories = {}
metrics = ["train_acc", "val_acc", "train_loss", "val_loss"]

for name, run_list in all_raw_histories.items():
    avg_hist = {}

    for m in metrics:
        vals = [np.array(h.get(m, []), dtype=float) for h in run_list if h.get(m, [])]
        if len(vals) > 0:
            stacked = np.stack(vals, axis=0)  # (n_runs, n_epochs)
            avg_hist[m] = np.mean(stacked, axis=0).tolist()
        else:
            avg_hist[m] = []

    final_test_accs = [h.get("final_test_acc", 0.0) for h in run_list]
    avg_hist["final_test_acc"] = float(np.mean(final_test_accs)) if len(final_test_accs) > 0 else 0.0

    averaged_histories[name] = avg_hist

print("\n" + "=" * 60)
print(f"VISUALIZING AVERAGE TRAINING METRICS ({N_RUNS} Runs)")
print("=" * 60)
visualize_training_comparisons(averaged_histories)


# ==========================================
# 6. Visualization: RSA Matrices (PER CONDITION labels)
# ==========================================
print("\n" + "=" * 60)
print(f"VISUALIZING TRAINING DATA MEAN RSA MATRICES ({N_RUNS} Runs)")
print("=" * 60)
aggregate_and_visualize_rsa(all_raw_matrices_train, train_labels_cache, title_suffix="[Train]")

print("\n" + "=" * 60)
print(f"VISUALIZING TESTING DATA MEAN RSA MATRICES ({N_RUNS} Runs)")
print("=" * 60)
aggregate_and_visualize_rsa(all_raw_matrices_test, test_labels_cache, title_suffix="[Test]")


# ==========================================
# 7. Final Summary Table
# ==========================================
print("\n" + "=" * 80)
print(f"FINAL AVERAGED RESULTS SUMMARY ({N_RUNS} Runs)")
print("=" * 80)
print(f"{'Condition':<30} | {'Train Acc':<12} | {'Val Acc':<12} | {'Test Acc':<12}")
print("-" * 80)

for name, hist in averaged_histories.items():
    train_acc_list = hist.get("train_acc", [])
    val_acc_list   = hist.get("val_acc", [])
    train_acc = train_acc_list[-1] if len(train_acc_list) > 0 else 0.0
    val_acc   = val_acc_list[-1] if len(val_acc_list) > 0 else 0.0
    test_acc  = hist.get("final_test_acc", 0.0)

    print(f"{name:<30} | {train_acc:>10.2f}% | {val_acc:>10.2f}% | {test_acc:>10.2f}%")

print("=" * 80)

# Optional: save final averaged summary to the last run directory (if N_RUNS==1, this is the only run)
# If you want a single global directory for all runs, set a separate path and save there.
if N_RUNS == 1:
    final_dir = os.path.join(SAVE_BASE, f"{EPOCHS}_0")
    save_json(averaged_histories, os.path.join(final_dir, "averaged_histories.json"))

Output hidden; open in https://colab.research.google.com to view.

### 8.4 VGGFace2 — 100 identities x 100 train + 10 test

Rebuild DataLoaders with RAM caching for faster iteration.


In [ ]:
# ==========================================
# 6. DataLoaders (RAM Caching Optimized)
# ==========================================

def cache_data_to_memory(dataset, batch_size=1024, desc="Caching"):
    """
    Reads the entire dataset from disk into a single in-memory TensorDataset.
    Using a temporary loader with workers speeds up the initial read.
    """
    print(f"Reading {desc} into RAM (High-Speed Optimization)...")
    # Use workers here to read from disk fast
    temp_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    all_images = []
    all_labels = []

    for imgs, lbls in tqdm(temp_loader):
        all_images.append(imgs)
        all_labels.append(lbls)

    # Create a TensorDataset (Pure RAM)
    return TensorDataset(torch.cat(all_images), torch.cat(all_labels))

# 1. Cache datasets into RAM
train_ram_dataset = cache_data_to_memory(train_set, desc="Train Data")
val_ram_dataset   = cache_data_to_memory(val_set,   desc="Val Data")
test_ram_dataset  = cache_data_to_memory(test_set, desc="Test Data")

# 2. Your Original Adapter (Unchanged Logic)
class ThreeValueAdapter(torch.utils.data.Dataset):
    def __init__(self, dataset):
        self.dataset = dataset
        # [Fix 1] Initialize class_to_idx as None, will be injected later
        self.class_to_idx = getattr(dataset, 'class_to_idx', None)

    def __getitem__(self, index):
        img, label = self.dataset[index]
        # Returns: Image, Label, Index (0, 1, 2)
        return img, label, index

    def __len__(self):
        return len(self.dataset)

# 3. Wrap the RAM datasets instead of the Disk datasets
train_dataset_wrapped = ThreeValueAdapter(train_ram_dataset)
val_dataset_wrapped   = ThreeValueAdapter(val_ram_dataset)
test_dataset_wrapped  = ThreeValueAdapter(test_ram_dataset)

# ==============================================================================
# The RSA code needs this map to convert ID 0 -> "n000001"
# ==============================================================================
if 'full_dataset' in globals() and hasattr(full_dataset, 'class_to_idx'):
    print(" Injecting class_to_idx map from full_dataset into RAM datasets...")
    mapping = full_dataset.class_to_idx

    # Inject into the Adapter
    train_dataset_wrapped.class_to_idx = mapping
    val_dataset_wrapped.class_to_idx   = mapping
    test_dataset_wrapped.class_to_idx  = mapping

    # Inject into the underlying TensorDataset
    train_ram_dataset.class_to_idx = mapping
    val_ram_dataset.class_to_idx   = mapping
    test_ram_dataset.class_to_idx  = mapping
    print(" Injection successful.")
else:
    print(" Warning: full_dataset.class_to_idx not found. RSA plotting might fail.")
# ==============================================================================

SAFE_BATCH_SIZE = 2048

# 4. Final Loaders
train_loader = DataLoader(train_dataset_wrapped, batch_size=SAFE_BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset_wrapped, batch_size=SAFE_BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset_wrapped, batch_size=SAFE_BATCH_SIZE, shuffle=False, num_workers=0)

loaders = {"train": train_loader, "val": val_loader, "test": test_loader}

print(f"DataLoaders ready.")
print(f"Structure: RAM TensorDataset -> ThreeValueAdapter -> DataLoader")

Reading Train Data into RAM (High-Speed Optimization)...


  0%|          | 0/8 [00:00<?, ?it/s]

Reading Val Data into RAM (High-Speed Optimization)...


  0%|          | 0/1 [00:00<?, ?it/s]

Reading Test Data into RAM (High-Speed Optimization)...


  0%|          | 0/1 [00:00<?, ?it/s]

 Injecting class_to_idx map from full_dataset into RAM datasets...
 Injection successful.
DataLoaders ready.
Structure: RAM TensorDataset -> ThreeValueAdapter -> DataLoader


#### 8.4a  decoder=64, dropout=0.0, 1 run x 100 epochs


In [ ]:
# ==========================================
# 1. Conditions & Hyperparams
# ==========================================
conditions = [
    {"alpha": 0.5, "noise": 0.0, "name": "Inhibitated"},
    {"alpha": 1.0, "noise": 0.0, "name": "Balanced"},
    {"alpha": 2.0, "noise": 0.0, "name": "Excitated"},
]
N_RUNS = 1
EPOCHS = 100


# ==========================================
# 2. Define loaders (train vs RSA)
# ==========================================
train_loader_train = loaders["train"]
val_loader_train   = loaders["val"]
test_loader_train  = loaders["test"]

# RSA loaders must be deterministic (shuffle=False)
train_loader_rsa = make_rsa_loader(train_loader_train)
val_loader_rsa   = make_rsa_loader(val_loader_train)
test_loader_rsa  = make_rsa_loader(test_loader_train)

# -------------------------------------------------------------
# [Safety Net] Double-check injection for RSA Analysis
# -------------------------------------------------------------
print(" Ensuring class_to_idx exists in RSA loaders...")

source_map = None
if 'full_dataset' in globals() and hasattr(full_dataset, 'class_to_idx'):
    source_map = full_dataset.class_to_idx
elif hasattr(train_loader_train.dataset, 'class_to_idx'):
    source_map = train_loader_train.dataset.class_to_idx

if source_map:
    train_loader_rsa.dataset.class_to_idx = source_map
    val_loader_rsa.dataset.class_to_idx   = source_map
    test_loader_rsa.dataset.class_to_idx  = source_map
    print(" Verified: 'class_to_idx' is ready for RSA plotting.")
else:
    print(" Warning: Could not find 'class_to_idx'. RSA labels might show as numbers.")
# -------------------------------------------------------------

class BatchAugmenter(nn.Module):
    def __init__(self):
        super().__init__()
        self.aug = nn.Sequential(
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.Pad(4, padding_mode='reflect'),
            transforms.RandomCrop(64),
        )

    def forward(self, x):
        return self.aug(x)

batch_augmenter = BatchAugmenter().to(DEVICE)
print("Batch Augmentation initialized.")
# ==========================================
# 3. Storage for Results
# ==========================================
all_raw_histories      = defaultdict(list)
all_raw_matrices_train = defaultdict(list)
all_raw_matrices_test  = defaultdict(list)

# Store labels PER CONDITION (prevents label/matrix misalignment across conditions)
train_labels_cache = {}   # {cond_name: labels_list}
test_labels_cache  = {}   # {cond_name: labels_list}

print(f"Starting {N_RUNS} runs per condition on {DEVICE}...")

from datetime import datetime

EXP_TIME = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

EXP_DIR = os.path.join(
    SAVE_BASE,
    f"{EPOCHS}_{N_RUNS}_{str(N_PEOPLE)}_{str(IMGS_PER_PERSON)}_{EXP_TIME}"
)
ensure_dir(EXP_DIR)

print(f"Experiment directory: {EXP_DIR}")
# ==========================================
# 4. Run Experiments (Training + RSA)
# ==========================================
for run_idx in tqdm(range(N_RUNS), desc="Total Progress"):

    run_dir = os.path.join(
        EXP_DIR,
        f"run_{run_idx}"
    )
    ensure_dir(run_dir)

    for cond in conditions:
        cond_name = cond["name"]
        cond_dir = os.path.join(run_dir, safe_name(cond_name))
        ensure_dir(cond_dir)

        # -------------------------------
        # A. Initialize Model
        # -------------------------------
        model = build_cornet_for_training(
            num_classes=N_PEOPLE,
            alpha=cond["alpha"],
            noise_std=cond["noise"],
            freeze_backbone=False,
            penultimate_dim=64,
            penultimate_dropout=0.0,
        )

        # -------------------------------
        # B. Train Model
        # -------------------------------
        _, history = train_cornet(
            model,
            train_loader=train_loader_train,
            val_loader=val_loader_train,
            test_loader=test_loader_train,
            epochs=EPOCHS,
            lr=1e-4,
            device=DEVICE,
            weight_decay=0,
            use_augmentation=False,
        )
        all_raw_histories[cond_name].append(history)

        # Save raw training history (per condition, per run)
        save_json(history, os.path.join(cond_dir, f"history_run{run_idx}.json"))

        # -------------------------------
        # C1. RSA on Training Set (RSA loader)
        # -------------------------------
        model.eval()

        matrix_train, label_indices_train = compute_rsa_matrix(
            model,
            train_loader_rsa,
            device=DEVICE,
            layer="penultimate_dense",
            metric="pearson",
        )
        all_raw_matrices_train[cond_name].append(matrix_train)

        labels_train_now = get_person_name_labels(
            train_loader_rsa,
            label_indices_train,
            add_image_number=True,
        )

        if cond_name not in train_labels_cache:
            train_labels_cache[cond_name] = labels_train_now
            print(f"[Train][{cond_name}] Labels example: {labels_train_now[:10]}")
        else:
            assert labels_train_now == train_labels_cache[cond_name], (
                f"[Train][{cond_name}] RSA sample order changed. "
                "Check shuffle/sampler or random augmentations in the RSA dataset."
            )

        # Save raw train RSA outputs (per condition, per run)
        save_numpy(matrix_train, os.path.join(cond_dir, f"rsa_train_run{run_idx}.npy"))
        save_json(labels_train_now, os.path.join(cond_dir, f"rsa_train_labels_run{run_idx}.json"))

        # -------------------------------
        # C2. RSA on Test Set (RSA loader)
        # -------------------------------
        matrix_test, label_indices_test = compute_rsa_matrix(
            model,
            test_loader_rsa,
            device=DEVICE,
            layer="penultimate_dense",
            metric="pearson",
        )
        all_raw_matrices_test[cond_name].append(matrix_test)

        labels_test_now = get_person_name_labels(
            test_loader_rsa,
            label_indices_test,
            add_image_number=True,
        )

        if cond_name not in test_labels_cache:
            test_labels_cache[cond_name] = labels_test_now
            print(f"[Test][{cond_name}] Labels example: {labels_test_now[:10]}")
        else:
            assert labels_test_now == test_labels_cache[cond_name], (
                f"[Test][{cond_name}] RSA sample order changed. "
                "Check shuffle/sampler or random augmentations in the RSA dataset."
            )

        # Save raw test RSA outputs (per condition, per run)
        save_numpy(matrix_test, os.path.join(cond_dir, f"rsa_test_run{run_idx}.npy"))
        save_json(labels_test_now, os.path.join(cond_dir, f"rsa_test_labels_run{run_idx}.json"))

        # -------------------------------
        # Save model checkpoint (per condition, per run)
        # -------------------------------
        ckpt_path = os.path.join(cond_dir, f"model_run{run_idx}.pt")
        ckpt_meta = {
            "seed": SEED,
            "device": str(DEVICE),
            "epochs": EPOCHS,
            "run_idx": run_idx,
            "condition": cond,
            "layer_for_rsa": "penultimate_dense",
        }
        save_model_checkpoint(model, ckpt_path, ckpt_meta)

        # -------------------------------
        # D. Cleanup GPU Memory
        # -------------------------------
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Save run-level raw dict snapshots (useful if you want one file per run)
    # Histories are JSON-serializable; matrices are saved as NPZ.
    run_histories_path = os.path.join(run_dir, "all_raw_histories.json")
    run_mats_train_path = os.path.join(run_dir, "all_raw_matrices_train.npz")
    run_mats_test_path  = os.path.join(run_dir, "all_raw_matrices_test.npz")
    run_labels_train_path = os.path.join(run_dir, "train_labels_by_condition.json")
    run_labels_test_path  = os.path.join(run_dir, "test_labels_by_condition.json")

    save_json(all_raw_histories, run_histories_path)
    save_json(train_labels_cache, run_labels_train_path)
    save_json(test_labels_cache, run_labels_test_path)

    # Save matrices per condition into one NPZ per split
    # Each condition becomes a separate NPZ file inside the run directory.
    for cond in conditions:
        cond_name = cond["name"]
        cond_safe = safe_name(cond_name)

        save_npz(
            all_raw_matrices_train[cond_name],
            os.path.join(run_dir, f"{cond_safe}_train_matrices.npz")
        )
        save_npz(
            all_raw_matrices_test[cond_name],
            os.path.join(run_dir, f"{cond_safe}_test_matrices.npz")
        )

print("All runs completed. Aggregating data...")

# ==========================================
# 5. Visualization: Training Dynamics
# ==========================================


Output hidden; open in https://colab.research.google.com to view.

#### 8.4b  decoder=128, dropout=0.2, 1 run x 100 epochs


In [ ]:
# ==========================================
# 1. Conditions & Hyperparams
# ==========================================
conditions = [
    {"alpha": 0.5, "noise": 0.0, "name": "Inhibitated"},
    {"alpha": 1.0, "noise": 0.0, "name": "Balanced"},
    {"alpha": 2.0, "noise": 0.0, "name": "Excitated"},
]
N_RUNS = 1
EPOCHS = 100


# ==========================================
# 2. Define loaders (train vs RSA)
# ==========================================
train_loader_train = loaders["train"]
val_loader_train   = loaders["val"]
test_loader_train  = loaders["test"]

# RSA loaders must be deterministic (shuffle=False)
train_loader_rsa = make_rsa_loader(train_loader_train)
val_loader_rsa   = make_rsa_loader(val_loader_train)
test_loader_rsa  = make_rsa_loader(test_loader_train)

# -------------------------------------------------------------
# [Safety Net] Double-check injection for RSA Analysis
# -------------------------------------------------------------
print(" Ensuring class_to_idx exists in RSA loaders...")

source_map = None
if 'full_dataset' in globals() and hasattr(full_dataset, 'class_to_idx'):
    source_map = full_dataset.class_to_idx
elif hasattr(train_loader_train.dataset, 'class_to_idx'):
    source_map = train_loader_train.dataset.class_to_idx

if source_map:
    train_loader_rsa.dataset.class_to_idx = source_map
    val_loader_rsa.dataset.class_to_idx   = source_map
    test_loader_rsa.dataset.class_to_idx  = source_map
    print(" Verified: 'class_to_idx' is ready for RSA plotting.")
else:
    print(" Warning: Could not find 'class_to_idx'. RSA labels might show as numbers.")
# -------------------------------------------------------------

class BatchAugmenter(nn.Module):
    def __init__(self):
        super().__init__()
        self.aug = nn.Sequential(
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.Pad(4, padding_mode='reflect'),
            transforms.RandomCrop(64),
        )

    def forward(self, x):
        return self.aug(x)

batch_augmenter = BatchAugmenter().to(DEVICE)
print("Batch Augmentation initialized.")
# ==========================================
# 3. Storage for Results
# ==========================================
all_raw_histories      = defaultdict(list)
all_raw_matrices_train = defaultdict(list)
all_raw_matrices_test  = defaultdict(list)

# Store labels PER CONDITION (prevents label/matrix misalignment across conditions)
train_labels_cache = {}   # {cond_name: labels_list}
test_labels_cache  = {}   # {cond_name: labels_list}

print(f"Starting {N_RUNS} runs per condition on {DEVICE}...")

from datetime import datetime

EXP_TIME = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

EXP_DIR = os.path.join(
    SAVE_BASE,
    f"{EPOCHS}_{N_RUNS}_{str(N_PEOPLE)}_{str(IMGS_PER_PERSON)}_{EXP_TIME}"
)
ensure_dir(EXP_DIR)

print(f"Experiment directory: {EXP_DIR}")
# ==========================================
# 4. Run Experiments (Training + RSA)
# ==========================================
for run_idx in tqdm(range(N_RUNS), desc="Total Progress"):

    run_dir = os.path.join(
        EXP_DIR,
        f"run_{run_idx}"
    )
    ensure_dir(run_dir)

    for cond in conditions:
        cond_name = cond["name"]
        cond_dir = os.path.join(run_dir, safe_name(cond_name))
        ensure_dir(cond_dir)

        # -------------------------------
        # A. Initialize Model
        # -------------------------------
        model = build_cornet_for_training(
            num_classes=N_PEOPLE,
            alpha=cond["alpha"],
            noise_std=cond["noise"],
            freeze_backbone=False,
            penultimate_dim=128,
            penultimate_dropout=0.2,
        )

        # -------------------------------
        # B. Train Model
        # -------------------------------
        _, history = train_cornet(
            model,
            train_loader=train_loader_train,
            val_loader=val_loader_train,
            test_loader=test_loader_train,
            epochs=EPOCHS,
            lr=1e-4,
            device=DEVICE,
            weight_decay=0,
            use_augmentation=False,
        )
        all_raw_histories[cond_name].append(history)

        # Save raw training history (per condition, per run)
        save_json(history, os.path.join(cond_dir, f"history_run{run_idx}.json"))

        # -------------------------------
        # C1. RSA on Training Set (RSA loader)
        # -------------------------------
        model.eval()

        matrix_train, label_indices_train = compute_rsa_matrix(
            model,
            train_loader_rsa,
            device=DEVICE,
            layer="penultimate_dense",
            metric="pearson",
        )
        all_raw_matrices_train[cond_name].append(matrix_train)

        labels_train_now = get_person_name_labels(
            train_loader_rsa,
            label_indices_train,
            add_image_number=True,
        )

        if cond_name not in train_labels_cache:
            train_labels_cache[cond_name] = labels_train_now
            print(f"[Train][{cond_name}] Labels example: {labels_train_now[:10]}")
        else:
            assert labels_train_now == train_labels_cache[cond_name], (
                f"[Train][{cond_name}] RSA sample order changed. "
                "Check shuffle/sampler or random augmentations in the RSA dataset."
            )

        # Save raw train RSA outputs (per condition, per run)
        save_numpy(matrix_train, os.path.join(cond_dir, f"rsa_train_run{run_idx}.npy"))
        save_json(labels_train_now, os.path.join(cond_dir, f"rsa_train_labels_run{run_idx}.json"))

        # -------------------------------
        # C2. RSA on Test Set (RSA loader)
        # -------------------------------
        matrix_test, label_indices_test = compute_rsa_matrix(
            model,
            test_loader_rsa,
            device=DEVICE,
            layer="penultimate_dense",
            metric="pearson",
        )
        all_raw_matrices_test[cond_name].append(matrix_test)

        labels_test_now = get_person_name_labels(
            test_loader_rsa,
            label_indices_test,
            add_image_number=True,
        )

        if cond_name not in test_labels_cache:
            test_labels_cache[cond_name] = labels_test_now
            print(f"[Test][{cond_name}] Labels example: {labels_test_now[:10]}")
        else:
            assert labels_test_now == test_labels_cache[cond_name], (
                f"[Test][{cond_name}] RSA sample order changed. "
                "Check shuffle/sampler or random augmentations in the RSA dataset."
            )

        # Save raw test RSA outputs (per condition, per run)
        save_numpy(matrix_test, os.path.join(cond_dir, f"rsa_test_run{run_idx}.npy"))
        save_json(labels_test_now, os.path.join(cond_dir, f"rsa_test_labels_run{run_idx}.json"))

        # -------------------------------
        # Save model checkpoint (per condition, per run)
        # -------------------------------
        ckpt_path = os.path.join(cond_dir, f"model_run{run_idx}.pt")
        ckpt_meta = {
            "seed": SEED,
            "device": str(DEVICE),
            "epochs": EPOCHS,
            "run_idx": run_idx,
            "condition": cond,
            "layer_for_rsa": "penultimate_dense",
        }
        save_model_checkpoint(model, ckpt_path, ckpt_meta)

        # -------------------------------
        # D. Cleanup GPU Memory
        # -------------------------------
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Save run-level raw dict snapshots (useful if you want one file per run)
    # Histories are JSON-serializable; matrices are saved as NPZ.
    run_histories_path = os.path.join(run_dir, "all_raw_histories.json")
    run_mats_train_path = os.path.join(run_dir, "all_raw_matrices_train.npz")
    run_mats_test_path  = os.path.join(run_dir, "all_raw_matrices_test.npz")
    run_labels_train_path = os.path.join(run_dir, "train_labels_by_condition.json")
    run_labels_test_path  = os.path.join(run_dir, "test_labels_by_condition.json")

    save_json(all_raw_histories, run_histories_path)
    save_json(train_labels_cache, run_labels_train_path)
    save_json(test_labels_cache, run_labels_test_path)

    # Save matrices per condition into one NPZ per split
    # Each condition becomes a separate NPZ file inside the run directory.
    for cond in conditions:
        cond_name = cond["name"]
        cond_safe = safe_name(cond_name)

        save_npz(
            all_raw_matrices_train[cond_name],
            os.path.join(run_dir, f"{cond_safe}_train_matrices.npz")
        )
        save_npz(
            all_raw_matrices_test[cond_name],
            os.path.join(run_dir, f"{cond_safe}_test_matrices.npz")
        )

print("All runs completed. Aggregating data...")



run_dir_path = os.path.join(
    "/content/drive/MyDrive/ASD_FaceReg_Modeling_CNN/results/EIB/cornet",
    EXP_DIR,
    "run_0"
)


summary_stats = visualize_local_results(
    run_dir=run_dir_path,
    conditions=["Balanced", "Excitated", "Inhibitated"],
    show_training_curves=True,
    show_rsa_matrices=True,
    full_label=False,
)


print("\nReturned Summary Data:")
print(summary_stats)

# Optional: save final averaged summary to the last run directory (if N_RUNS==1, this is the only run)
# If you want a single global directory for all runs, set a separate path and save there.
if N_RUNS == 1:
    final_dir = os.path.join(SAVE_BASE, f"{EPOCHS}_0")
    save_json(averaged_histories, os.path.join(final_dir, "averaged_histories.json"))

Output hidden; open in https://colab.research.google.com to view.

#### 8.4c  decoder=256, dropout=0.1, 1 run x 80 epochs


In [ ]:
# ==========================================
# 1. Conditions & Hyperparams
# ==========================================
conditions = [
    {"alpha": 0.5, "noise": 0.0, "name": "Inhibitated"},
    {"alpha": 1.0, "noise": 0.0, "name": "Balanced"},
    {"alpha": 2.0, "noise": 0.0, "name": "Excitated"},
]
N_RUNS = 1
EPOCHS = 80


# ==========================================
# 2. Define loaders (train vs RSA)
# ==========================================
train_loader_train = loaders["train"]
val_loader_train   = loaders["val"]
test_loader_train  = loaders["test"]

# RSA loaders must be deterministic (shuffle=False)
train_loader_rsa = make_rsa_loader(train_loader_train)
val_loader_rsa   = make_rsa_loader(val_loader_train)
test_loader_rsa  = make_rsa_loader(test_loader_train)

# -------------------------------------------------------------
# [Safety Net] Double-check injection for RSA Analysis
# -------------------------------------------------------------
print(" Ensuring class_to_idx exists in RSA loaders...")

source_map = None
if 'full_dataset' in globals() and hasattr(full_dataset, 'class_to_idx'):
    source_map = full_dataset.class_to_idx
elif hasattr(train_loader_train.dataset, 'class_to_idx'):
    source_map = train_loader_train.dataset.class_to_idx

if source_map:
    train_loader_rsa.dataset.class_to_idx = source_map
    val_loader_rsa.dataset.class_to_idx   = source_map
    test_loader_rsa.dataset.class_to_idx  = source_map
    print(" Verified: 'class_to_idx' is ready for RSA plotting.")
else:
    print(" Warning: Could not find 'class_to_idx'. RSA labels might show as numbers.")
# -------------------------------------------------------------

class BatchAugmenter(nn.Module):
    def __init__(self):
        super().__init__()
        self.aug = nn.Sequential(
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.Pad(4, padding_mode='reflect'),
            transforms.RandomCrop(64),
        )

    def forward(self, x):
        return self.aug(x)

batch_augmenter = BatchAugmenter().to(DEVICE)
print("Batch Augmentation initialized.")
# ==========================================
# 3. Storage for Results
# ==========================================
all_raw_histories      = defaultdict(list)
all_raw_matrices_train = defaultdict(list)
all_raw_matrices_test  = defaultdict(list)

# Store labels PER CONDITION (prevents label/matrix misalignment across conditions)
train_labels_cache = {}   # {cond_name: labels_list}
test_labels_cache  = {}   # {cond_name: labels_list}

print(f"Starting {N_RUNS} runs per condition on {DEVICE}...")

from datetime import datetime

EXP_TIME = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

EXP_DIR = os.path.join(
    SAVE_BASE,
    f"{EPOCHS}_{N_RUNS}_{str(N_PEOPLE)}_{str(IMGS_PER_PERSON)}_{EXP_TIME}"
)
ensure_dir(EXP_DIR)

print(f"Experiment directory: {EXP_DIR}")
# ==========================================
# 4. Run Experiments (Training + RSA)
# ==========================================
for run_idx in tqdm(range(N_RUNS), desc="Total Progress"):

    run_dir = os.path.join(
        EXP_DIR,
        f"run_{run_idx}"
    )
    ensure_dir(run_dir)

    for cond in conditions:
        cond_name = cond["name"]
        cond_dir = os.path.join(run_dir, safe_name(cond_name))
        ensure_dir(cond_dir)

        # -------------------------------
        # A. Initialize Model
        # -------------------------------
        model = build_cornet_for_training(
            num_classes=N_PEOPLE,
            alpha=cond["alpha"],
            noise_std=cond["noise"],
            freeze_backbone=False,
            penultimate_dim=256,
            penultimate_dropout=0.1,
        )

        # -------------------------------
        # B. Train Model
        # -------------------------------
        _, history = train_cornet(
            model,
            train_loader=train_loader_train,
            val_loader=val_loader_train,
            test_loader=test_loader_train,
            epochs=EPOCHS,
            lr=1e-4,
            device=DEVICE,
            weight_decay=0,
            use_augmentation=False,
        )
        all_raw_histories[cond_name].append(history)

        # Save raw training history (per condition, per run)
        save_json(history, os.path.join(cond_dir, f"history_run{run_idx}.json"))

        # -------------------------------
        # C1. RSA on Training Set (RSA loader)
        # -------------------------------
        model.eval()

        matrix_train, label_indices_train = compute_rsa_matrix(
            model,
            train_loader_rsa,
            device=DEVICE,
            layer="penultimate_dense",
            metric="pearson",
        )
        all_raw_matrices_train[cond_name].append(matrix_train)

        labels_train_now = get_person_name_labels(
            train_loader_rsa,
            label_indices_train,
            add_image_number=True,
        )

        if cond_name not in train_labels_cache:
            train_labels_cache[cond_name] = labels_train_now
            print(f"[Train][{cond_name}] Labels example: {labels_train_now[:10]}")
        else:
            assert labels_train_now == train_labels_cache[cond_name], (
                f"[Train][{cond_name}] RSA sample order changed. "
                "Check shuffle/sampler or random augmentations in the RSA dataset."
            )

        # Save raw train RSA outputs (per condition, per run)
        save_numpy(matrix_train, os.path.join(cond_dir, f"rsa_train_run{run_idx}.npy"))
        save_json(labels_train_now, os.path.join(cond_dir, f"rsa_train_labels_run{run_idx}.json"))

        # -------------------------------
        # C2. RSA on Test Set (RSA loader)
        # -------------------------------
        matrix_test, label_indices_test = compute_rsa_matrix(
            model,
            test_loader_rsa,
            device=DEVICE,
            layer="penultimate_dense",
            metric="pearson",
        )
        all_raw_matrices_test[cond_name].append(matrix_test)

        labels_test_now = get_person_name_labels(
            test_loader_rsa,
            label_indices_test,
            add_image_number=True,
        )

        if cond_name not in test_labels_cache:
            test_labels_cache[cond_name] = labels_test_now
            print(f"[Test][{cond_name}] Labels example: {labels_test_now[:10]}")
        else:
            assert labels_test_now == test_labels_cache[cond_name], (
                f"[Test][{cond_name}] RSA sample order changed. "
                "Check shuffle/sampler or random augmentations in the RSA dataset."
            )

        # Save raw test RSA outputs (per condition, per run)
        save_numpy(matrix_test, os.path.join(cond_dir, f"rsa_test_run{run_idx}.npy"))
        save_json(labels_test_now, os.path.join(cond_dir, f"rsa_test_labels_run{run_idx}.json"))

        # -------------------------------
        # Save model checkpoint (per condition, per run)
        # -------------------------------
        ckpt_path = os.path.join(cond_dir, f"model_run{run_idx}.pt")
        ckpt_meta = {
            "seed": SEED,
            "device": str(DEVICE),
            "epochs": EPOCHS,
            "run_idx": run_idx,
            "condition": cond,
            "layer_for_rsa": "penultimate_dense",
        }
        save_model_checkpoint(model, ckpt_path, ckpt_meta)

        # -------------------------------
        # D. Cleanup GPU Memory
        # -------------------------------
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Save run-level raw dict snapshots (useful if you want one file per run)
    # Histories are JSON-serializable; matrices are saved as NPZ.
    run_histories_path = os.path.join(run_dir, "all_raw_histories.json")
    run_mats_train_path = os.path.join(run_dir, "all_raw_matrices_train.npz")
    run_mats_test_path  = os.path.join(run_dir, "all_raw_matrices_test.npz")
    run_labels_train_path = os.path.join(run_dir, "train_labels_by_condition.json")
    run_labels_test_path  = os.path.join(run_dir, "test_labels_by_condition.json")

    save_json(all_raw_histories, run_histories_path)
    save_json(train_labels_cache, run_labels_train_path)
    save_json(test_labels_cache, run_labels_test_path)

    # Save matrices per condition into one NPZ per split
    # Each condition becomes a separate NPZ file inside the run directory.
    for cond in conditions:
        cond_name = cond["name"]
        cond_safe = safe_name(cond_name)

        save_npz(
            all_raw_matrices_train[cond_name],
            os.path.join(run_dir, f"{cond_safe}_train_matrices.npz")
        )
        save_npz(
            all_raw_matrices_test[cond_name],
            os.path.join(run_dir, f"{cond_safe}_test_matrices.npz")
        )

print("All runs completed. Aggregating data...")

# ==========================================
# 5. Visualization: Training Dynamics
# ==========================================
averaged_histories = {}
metrics = ["train_acc", "val_acc", "train_loss", "val_loss"]

for name, run_list in all_raw_histories.items():
    avg_hist = {}

    for m in metrics:
        vals = [np.array(h.get(m, []), dtype=float) for h in run_list if h.get(m, [])]
        if len(vals) > 0:
            stacked = np.stack(vals, axis=0)  # (n_runs, n_epochs)
            avg_hist[m] = np.mean(stacked, axis=0).tolist()
        else:
            avg_hist[m] = []

    final_test_accs = [h.get("final_test_acc", 0.0) for h in run_list]
    avg_hist["final_test_acc"] = float(np.mean(final_test_accs)) if len(final_test_accs) > 0 else 0.0

    averaged_histories[name] = avg_hist

print("\n" + "=" * 60)
print(f"VISUALIZING AVERAGE TRAINING METRICS ({N_RUNS} Runs)")
print("=" * 60)
visualize_training_comparisons(averaged_histories)


# ==========================================
# 6. Visualization: RSA Matrices (PER CONDITION labels)
# ==========================================
print("\n" + "=" * 60)
print(f"VISUALIZING TRAINING DATA MEAN RSA MATRICES ({N_RUNS} Runs)")
print("=" * 60)
aggregate_and_visualize_rsa(all_raw_matrices_train, train_labels_cache, title_suffix="[Train]")

print("\n" + "=" * 60)
print(f"VISUALIZING TESTING DATA MEAN RSA MATRICES ({N_RUNS} Runs)")
print("=" * 60)
aggregate_and_visualize_rsa(all_raw_matrices_test, test_labels_cache, title_suffix="[Test]")


# ==========================================
# 7. Final Summary Table
# ==========================================
print("\n" + "=" * 80)
print(f"FINAL AVERAGED RESULTS SUMMARY ({N_RUNS} Runs)")
print("=" * 80)
print(f"{'Condition':<30} | {'Train Acc':<12} | {'Val Acc':<12} | {'Test Acc':<12}")
print("-" * 80)

for name, hist in averaged_histories.items():
    train_acc_list = hist.get("train_acc", [])
    val_acc_list   = hist.get("val_acc", [])
    train_acc = train_acc_list[-1] if len(train_acc_list) > 0 else 0.0
    val_acc   = val_acc_list[-1] if len(val_acc_list) > 0 else 0.0
    test_acc  = hist.get("final_test_acc", 0.0)

    print(f"{name:<30} | {train_acc:>10.2f}% | {val_acc:>10.2f}% | {test_acc:>10.2f}%")

print("=" * 80)

# Optional: save final averaged summary to the last run directory (if N_RUNS==1, this is the only run)
# If you want a single global directory for all runs, set a separate path and save there.
if N_RUNS == 1:
    final_dir = os.path.join(SAVE_BASE, f"{EPOCHS}_0")
    save_json(averaged_histories, os.path.join(final_dir, "averaged_histories.json"))

Output hidden; open in https://colab.research.google.com to view.

#### 8.4d  decoder=256, dropout=0.2, 1 run x 80 epochs


In [ ]:
# ==========================================
# 1. Conditions & Hyperparams
# ==========================================
conditions = [
    {"alpha": 0.5, "noise": 0.0, "name": "Inhibitated"},
    {"alpha": 1.0, "noise": 0.0, "name": "Balanced"},
    {"alpha": 2.0, "noise": 0.0, "name": "Excitated"},
]
N_RUNS = 1
EPOCHS = 80


# ==========================================
# 2. Define loaders (train vs RSA)
# ==========================================
train_loader_train = loaders["train"]
val_loader_train   = loaders["val"]
test_loader_train  = loaders["test"]

# RSA loaders must be deterministic (shuffle=False)
train_loader_rsa = make_rsa_loader(train_loader_train)
val_loader_rsa   = make_rsa_loader(val_loader_train)
test_loader_rsa  = make_rsa_loader(test_loader_train)

# -------------------------------------------------------------
# [Safety Net] Double-check injection for RSA Analysis
# -------------------------------------------------------------
print(" Ensuring class_to_idx exists in RSA loaders...")

source_map = None
if 'full_dataset' in globals() and hasattr(full_dataset, 'class_to_idx'):
    source_map = full_dataset.class_to_idx
elif hasattr(train_loader_train.dataset, 'class_to_idx'):
    source_map = train_loader_train.dataset.class_to_idx

if source_map:
    train_loader_rsa.dataset.class_to_idx = source_map
    val_loader_rsa.dataset.class_to_idx   = source_map
    test_loader_rsa.dataset.class_to_idx  = source_map
    print(" Verified: 'class_to_idx' is ready for RSA plotting.")
else:
    print(" Warning: Could not find 'class_to_idx'. RSA labels might show as numbers.")
# -------------------------------------------------------------

class BatchAugmenter(nn.Module):
    def __init__(self):
        super().__init__()
        self.aug = nn.Sequential(
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.Pad(4, padding_mode='reflect'),
            transforms.RandomCrop(64),
        )

    def forward(self, x):
        return self.aug(x)

batch_augmenter = BatchAugmenter().to(DEVICE)
print("Batch Augmentation initialized.")
# ==========================================
# 3. Storage for Results
# ==========================================
all_raw_histories      = defaultdict(list)
all_raw_matrices_train = defaultdict(list)
all_raw_matrices_test  = defaultdict(list)

# Store labels PER CONDITION (prevents label/matrix misalignment across conditions)
train_labels_cache = {}   # {cond_name: labels_list}
test_labels_cache  = {}   # {cond_name: labels_list}

print(f"Starting {N_RUNS} runs per condition on {DEVICE}...")

from datetime import datetime

EXP_TIME = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

EXP_DIR = os.path.join(
    SAVE_BASE,
    f"{EPOCHS}_{N_RUNS}_{str(N_PEOPLE)}_{str(IMGS_PER_PERSON)}_{EXP_TIME}"
)
ensure_dir(EXP_DIR)

print(f"Experiment directory: {EXP_DIR}")
# ==========================================
# 4. Run Experiments (Training + RSA)
# ==========================================
for run_idx in tqdm(range(N_RUNS), desc="Total Progress"):

    run_dir = os.path.join(
        EXP_DIR,
        f"run_{run_idx}"
    )
    ensure_dir(run_dir)

    for cond in conditions:
        cond_name = cond["name"]
        cond_dir = os.path.join(run_dir, safe_name(cond_name))
        ensure_dir(cond_dir)

        # -------------------------------
        # A. Initialize Model
        # -------------------------------
        model = build_cornet_for_training(
            num_classes=N_PEOPLE,
            alpha=cond["alpha"],
            noise_std=cond["noise"],
            freeze_backbone=False,
            penultimate_dim=256,
            penultimate_dropout=0.2,
        )

        # -------------------------------
        # B. Train Model
        # -------------------------------
        _, history = train_cornet(
            model,
            train_loader=train_loader_train,
            val_loader=val_loader_train,
            test_loader=test_loader_train,
            epochs=EPOCHS,
            lr=1e-4,
            device=DEVICE,
            weight_decay=0,
            use_augmentation=False,
        )
        all_raw_histories[cond_name].append(history)

        # Save raw training history (per condition, per run)
        save_json(history, os.path.join(cond_dir, f"history_run{run_idx}.json"))

        # -------------------------------
        # C1. RSA on Training Set (RSA loader)
        # -------------------------------
        model.eval()

        matrix_train, label_indices_train = compute_rsa_matrix(
            model,
            train_loader_rsa,
            device=DEVICE,
            layer="penultimate_dense",
            metric="pearson",
        )
        all_raw_matrices_train[cond_name].append(matrix_train)

        labels_train_now = get_person_name_labels(
            train_loader_rsa,
            label_indices_train,
            add_image_number=True,
        )

        if cond_name not in train_labels_cache:
            train_labels_cache[cond_name] = labels_train_now
            print(f"[Train][{cond_name}] Labels example: {labels_train_now[:10]}")
        else:
            assert labels_train_now == train_labels_cache[cond_name], (
                f"[Train][{cond_name}] RSA sample order changed. "
                "Check shuffle/sampler or random augmentations in the RSA dataset."
            )

        # Save raw train RSA outputs (per condition, per run)
        save_numpy(matrix_train, os.path.join(cond_dir, f"rsa_train_run{run_idx}.npy"))
        save_json(labels_train_now, os.path.join(cond_dir, f"rsa_train_labels_run{run_idx}.json"))

        # -------------------------------
        # C2. RSA on Test Set (RSA loader)
        # -------------------------------
        matrix_test, label_indices_test = compute_rsa_matrix(
            model,
            test_loader_rsa,
            device=DEVICE,
            layer="penultimate_dense",
            metric="pearson",
        )
        all_raw_matrices_test[cond_name].append(matrix_test)

        labels_test_now = get_person_name_labels(
            test_loader_rsa,
            label_indices_test,
            add_image_number=True,
        )

        if cond_name not in test_labels_cache:
            test_labels_cache[cond_name] = labels_test_now
            print(f"[Test][{cond_name}] Labels example: {labels_test_now[:10]}")
        else:
            assert labels_test_now == test_labels_cache[cond_name], (
                f"[Test][{cond_name}] RSA sample order changed. "
                "Check shuffle/sampler or random augmentations in the RSA dataset."
            )

        # Save raw test RSA outputs (per condition, per run)
        save_numpy(matrix_test, os.path.join(cond_dir, f"rsa_test_run{run_idx}.npy"))
        save_json(labels_test_now, os.path.join(cond_dir, f"rsa_test_labels_run{run_idx}.json"))

        # -------------------------------
        # Save model checkpoint (per condition, per run)
        # -------------------------------
        ckpt_path = os.path.join(cond_dir, f"model_run{run_idx}.pt")
        ckpt_meta = {
            "seed": SEED,
            "device": str(DEVICE),
            "epochs": EPOCHS,
            "run_idx": run_idx,
            "condition": cond,
            "layer_for_rsa": "penultimate_dense",
        }
        save_model_checkpoint(model, ckpt_path, ckpt_meta)

        # -------------------------------
        # D. Cleanup GPU Memory
        # -------------------------------
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Save run-level raw dict snapshots (useful if you want one file per run)
    # Histories are JSON-serializable; matrices are saved as NPZ.
    run_histories_path = os.path.join(run_dir, "all_raw_histories.json")
    run_mats_train_path = os.path.join(run_dir, "all_raw_matrices_train.npz")
    run_mats_test_path  = os.path.join(run_dir, "all_raw_matrices_test.npz")
    run_labels_train_path = os.path.join(run_dir, "train_labels_by_condition.json")
    run_labels_test_path  = os.path.join(run_dir, "test_labels_by_condition.json")

    save_json(all_raw_histories, run_histories_path)
    save_json(train_labels_cache, run_labels_train_path)
    save_json(test_labels_cache, run_labels_test_path)

    # Save matrices per condition into one NPZ per split
    # Each condition becomes a separate NPZ file inside the run directory.
    for cond in conditions:
        cond_name = cond["name"]
        cond_safe = safe_name(cond_name)

        save_npz(
            all_raw_matrices_train[cond_name],
            os.path.join(run_dir, f"{cond_safe}_train_matrices.npz")
        )
        save_npz(
            all_raw_matrices_test[cond_name],
            os.path.join(run_dir, f"{cond_safe}_test_matrices.npz")
        )

print("All runs completed. Aggregating data...")

# ==========================================
# 5. Visualization: Training Dynamics
# ==========================================
averaged_histories = {}
metrics = ["train_acc", "val_acc", "train_loss", "val_loss"]

for name, run_list in all_raw_histories.items():
    avg_hist = {}

    for m in metrics:
        vals = [np.array(h.get(m, []), dtype=float) for h in run_list if h.get(m, [])]
        if len(vals) > 0:
            stacked = np.stack(vals, axis=0)  # (n_runs, n_epochs)
            avg_hist[m] = np.mean(stacked, axis=0).tolist()
        else:
            avg_hist[m] = []

    final_test_accs = [h.get("final_test_acc", 0.0) for h in run_list]
    avg_hist["final_test_acc"] = float(np.mean(final_test_accs)) if len(final_test_accs) > 0 else 0.0

    averaged_histories[name] = avg_hist

print("\n" + "=" * 60)
print(f"VISUALIZING AVERAGE TRAINING METRICS ({N_RUNS} Runs)")
print("=" * 60)
visualize_training_comparisons(averaged_histories)


# ==========================================
# 6. Visualization: RSA Matrices (PER CONDITION labels)
# ==========================================
print("\n" + "=" * 60)
print(f"VISUALIZING TRAINING DATA MEAN RSA MATRICES ({N_RUNS} Runs)")
print("=" * 60)
aggregate_and_visualize_rsa(all_raw_matrices_train, train_labels_cache, title_suffix="[Train]")

print("\n" + "=" * 60)
print(f"VISUALIZING TESTING DATA MEAN RSA MATRICES ({N_RUNS} Runs)")
print("=" * 60)
aggregate_and_visualize_rsa(all_raw_matrices_test, test_labels_cache, title_suffix="[Test]")


# ==========================================
# 7. Final Summary Table
# ==========================================
print("\n" + "=" * 80)
print(f"FINAL AVERAGED RESULTS SUMMARY ({N_RUNS} Runs)")
print("=" * 80)
print(f"{'Condition':<30} | {'Train Acc':<12} | {'Val Acc':<12} | {'Test Acc':<12}")
print("-" * 80)

for name, hist in averaged_histories.items():
    train_acc_list = hist.get("train_acc", [])
    val_acc_list   = hist.get("val_acc", [])
    train_acc = train_acc_list[-1] if len(train_acc_list) > 0 else 0.0
    val_acc   = val_acc_list[-1] if len(val_acc_list) > 0 else 0.0
    test_acc  = hist.get("final_test_acc", 0.0)

    print(f"{name:<30} | {train_acc:>10.2f}% | {val_acc:>10.2f}% | {test_acc:>10.2f}%")

print("=" * 80)

# Optional: save final averaged summary to the last run directory (if N_RUNS==1, this is the only run)
# If you want a single global directory for all runs, set a separate path and save there.
if N_RUNS == 1:
    final_dir = os.path.join(SAVE_BASE, f"{EPOCHS}_0")
    save_json(averaged_histories, os.path.join(final_dir, "averaged_histories.json"))

Output hidden; open in https://colab.research.google.com to view.

#### Colab Session Cleanup


In [ ]:
import time
from google.colab import runtime

print("✅ 所有任务已完成。")
print("⏳ 正在保存更改并准备断开连接 (10秒后)...")
time.sleep(10)

print("👋 晚安！正在断开 Runtime...")
runtime.unassign()

✅ 所有任务已完成。
⏳ 正在保存更改并准备断开连接 (10秒后)...
👋 晚安！正在断开 Runtime...


### 8.5 Selected Config — decoder=128, dropout=0.2, 20 runs x 100 epochs

Full-scale run with the decoder width and dropout chosen from the pilot sweeps above.


In [ ]:
# ==========================================
# 1. Conditions & Hyperparams
# ==========================================
conditions = [
    {"alpha": 0.5, "noise": 0.0, "name": "Inhibitated"},
    {"alpha": 1.0, "noise": 0.0, "name": "Balanced"},
    {"alpha": 2.0, "noise": 0.0, "name": "Excitated"},
]
N_RUNS = 20
EPOCHS = 100


# ==========================================
# 2. Define loaders (train vs RSA)
# ==========================================
train_loader_train = loaders["train"]
val_loader_train   = loaders["val"]
test_loader_train  = loaders["test"]

# RSA loaders must be deterministic (shuffle=False)
train_loader_rsa = make_rsa_loader(train_loader_train)
val_loader_rsa   = make_rsa_loader(val_loader_train)
test_loader_rsa  = make_rsa_loader(test_loader_train)

# -------------------------------------------------------------
# [Safety Net] Double-check injection for RSA Analysis
# -------------------------------------------------------------
print(" Ensuring class_to_idx exists in RSA loaders...")

source_map = None
if 'full_dataset' in globals() and hasattr(full_dataset, 'class_to_idx'):
    source_map = full_dataset.class_to_idx
elif hasattr(train_loader_train.dataset, 'class_to_idx'):
    source_map = train_loader_train.dataset.class_to_idx

if source_map:
    train_loader_rsa.dataset.class_to_idx = source_map
    val_loader_rsa.dataset.class_to_idx   = source_map
    test_loader_rsa.dataset.class_to_idx  = source_map
    print(" Verified: 'class_to_idx' is ready for RSA plotting.")
else:
    print(" Warning: Could not find 'class_to_idx'. RSA labels might show as numbers.")
# -------------------------------------------------------------

class BatchAugmenter(nn.Module):
    def __init__(self):
        super().__init__()
        self.aug = nn.Sequential(
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.Pad(4, padding_mode='reflect'),
            transforms.RandomCrop(64),
        )

    def forward(self, x):
        return self.aug(x)

batch_augmenter = BatchAugmenter().to(DEVICE)
print("Batch Augmentation initialized.")
# ==========================================
# 3. Storage for Results
# ==========================================
all_raw_histories      = defaultdict(list)
all_raw_matrices_train = defaultdict(list)
all_raw_matrices_test  = defaultdict(list)

# Store labels PER CONDITION (prevents label/matrix misalignment across conditions)
train_labels_cache = {}   # {cond_name: labels_list}
test_labels_cache  = {}   # {cond_name: labels_list}

print(f"Starting {N_RUNS} runs per condition on {DEVICE}...")

from datetime import datetime

EXP_TIME = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

EXP_DIR = os.path.join(
    SAVE_BASE,
    f"{EPOCHS}_{N_RUNS}_{str(N_PEOPLE)}_{str(IMGS_PER_PERSON)}_{EXP_TIME}"
)
ensure_dir(EXP_DIR)

print(f"Experiment directory: {EXP_DIR}")
# ==========================================
# 4. Run Experiments (Training + RSA)
# ==========================================
for run_idx in tqdm(range(N_RUNS), desc="Total Progress"):

    run_dir = os.path.join(
        EXP_DIR,
        f"run_{run_idx}"
    )
    ensure_dir(run_dir)

    for cond in conditions:
        cond_name = cond["name"]
        cond_dir = os.path.join(run_dir, safe_name(cond_name))
        ensure_dir(cond_dir)

        # -------------------------------
        # A. Initialize Model
        # -------------------------------
        model = build_cornet_for_training(
            num_classes=N_PEOPLE,
            alpha=cond["alpha"],
            noise_std=cond["noise"],
            freeze_backbone=False,
            penultimate_dim=128,
            penultimate_dropout=0.2,
        )

        # -------------------------------
        # B. Train Model
        # -------------------------------
        _, history = train_cornet(
            model,
            train_loader=train_loader_train,
            val_loader=val_loader_train,
            test_loader=test_loader_train,
            epochs=EPOCHS,
            lr=1e-4,
            device=DEVICE,
            weight_decay=0,
            use_augmentation=False,
        )
        all_raw_histories[cond_name].append(history)

        # Save raw training history (per condition, per run)
        save_json(history, os.path.join(cond_dir, f"history_run{run_idx}.json"))

        # -------------------------------
        # C1. RSA on Training Set (RSA loader)
        # -------------------------------
        model.eval()

        matrix_train, label_indices_train = compute_rsa_matrix(
            model,
            train_loader_rsa,
            device=DEVICE,
            layer="penultimate_dense",
            metric="pearson",
        )
        all_raw_matrices_train[cond_name].append(matrix_train)

        labels_train_now = get_person_name_labels(
            train_loader_rsa,
            label_indices_train,
            add_image_number=True,
        )

        if cond_name not in train_labels_cache:
            train_labels_cache[cond_name] = labels_train_now
            print(f"[Train][{cond_name}] Labels example: {labels_train_now[:10]}")
        else:
            assert labels_train_now == train_labels_cache[cond_name], (
                f"[Train][{cond_name}] RSA sample order changed. "
                "Check shuffle/sampler or random augmentations in the RSA dataset."
            )

        # Save raw train RSA outputs (per condition, per run)
        save_numpy(matrix_train, os.path.join(cond_dir, f"rsa_train_run{run_idx}.npy"))
        save_json(labels_train_now, os.path.join(cond_dir, f"rsa_train_labels_run{run_idx}.json"))

        # -------------------------------
        # C2. RSA on Test Set (RSA loader)
        # -------------------------------
        matrix_test, label_indices_test = compute_rsa_matrix(
            model,
            test_loader_rsa,
            device=DEVICE,
            layer="penultimate_dense",
            metric="pearson",
        )
        all_raw_matrices_test[cond_name].append(matrix_test)

        labels_test_now = get_person_name_labels(
            test_loader_rsa,
            label_indices_test,
            add_image_number=True,
        )

        if cond_name not in test_labels_cache:
            test_labels_cache[cond_name] = labels_test_now
            print(f"[Test][{cond_name}] Labels example: {labels_test_now[:10]}")
        else:
            assert labels_test_now == test_labels_cache[cond_name], (
                f"[Test][{cond_name}] RSA sample order changed. "
                "Check shuffle/sampler or random augmentations in the RSA dataset."
            )

        # Save raw test RSA outputs (per condition, per run)
        save_numpy(matrix_test, os.path.join(cond_dir, f"rsa_test_run{run_idx}.npy"))
        save_json(labels_test_now, os.path.join(cond_dir, f"rsa_test_labels_run{run_idx}.json"))

        # -------------------------------
        # Save model checkpoint (per condition, per run)
        # -------------------------------
        ckpt_path = os.path.join(cond_dir, f"model_run{run_idx}.pt")
        ckpt_meta = {
            "seed": SEED,
            "device": str(DEVICE),
            "epochs": EPOCHS,
            "run_idx": run_idx,
            "condition": cond,
            "layer_for_rsa": "penultimate_dense",
        }
        save_model_checkpoint(model, ckpt_path, ckpt_meta)

        # -------------------------------
        # D. Cleanup GPU Memory
        # -------------------------------
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Save run-level raw dict snapshots (useful if you want one file per run)
    # Histories are JSON-serializable; matrices are saved as NPZ.
    run_histories_path = os.path.join(run_dir, "all_raw_histories.json")
    run_mats_train_path = os.path.join(run_dir, "all_raw_matrices_train.npz")
    run_mats_test_path  = os.path.join(run_dir, "all_raw_matrices_test.npz")
    run_labels_train_path = os.path.join(run_dir, "train_labels_by_condition.json")
    run_labels_test_path  = os.path.join(run_dir, "test_labels_by_condition.json")

    save_json(all_raw_histories, run_histories_path)
    save_json(train_labels_cache, run_labels_train_path)
    save_json(test_labels_cache, run_labels_test_path)

    # Save matrices per condition into one NPZ per split
    # Each condition becomes a separate NPZ file inside the run directory.
    for cond in conditions:
        cond_name = cond["name"]
        cond_safe = safe_name(cond_name)

        save_npz(
            all_raw_matrices_train[cond_name],
            os.path.join(run_dir, f"{cond_safe}_train_matrices.npz")
        )
        save_npz(
            all_raw_matrices_test[cond_name],
            os.path.join(run_dir, f"{cond_safe}_test_matrices.npz")
        )

print("All runs completed. Aggregating data...")



run_dir_path = os.path.join(
    "/content/drive/MyDrive/ASD_FaceReg_Modeling_CNN/results/EIB/cornet",
    EXP_DIR,
    "run_0"
)


summary_stats = visualize_local_results(
    run_dir=run_dir_path,
    conditions=["Balanced", "Excitated", "Inhibitated"],
    show_training_curves=True,
    show_rsa_matrices=True,
    full_label=False,
)


print("\nReturned Summary Data:")
print(summary_stats)

# Optional: save final averaged summary to the last run directory (if N_RUNS==1, this is the only run)
# If you want a single global directory for all runs, set a separate path and save there.
if N_RUNS == 1:
    final_dir = os.path.join(SAVE_BASE, f"{EPOCHS}_0")
    save_json(averaged_histories, os.path.join(final_dir, "averaged_histories.json"))

Output hidden; open in https://colab.research.google.com to view.

#### Colab Session Cleanup


In [ ]:
import time
from google.colab import runtime

print("✅ 所有任务已完成。")
print("⏳ 正在保存更改并准备断开连接 (10秒后)...")
time.sleep(10)

print("👋 晚安！正在断开 Runtime...")
runtime.unassign()

✅ 所有任务已完成。
⏳ 正在保存更改并准备断开连接 (10秒后)...
👋 晚安！正在断开 Runtime...
